# Complexity-03k — L'arbre couvrant de poids minimal en temps linéaire déterministe (C#)

> **Bandeau preprint.** Ce carnet travaille sur le rapport ECCC TR26-339, *A Deterministic Linear Time Algorithm for Minimum Spanning Tree*, de Sanjeev Khanna, Aaron Putterman et Junkai Song, daté du 23 novembre 2026. C'est un **preprint, non évalué par les pairs** au moment de la rédaction. Le carnet **ne démontre pas** son résultat : il mesure les objets que ce résultat met en jeu, et sépare à chaque section ce qui est **mesuré ici**, ce qui est **cité du texte** des auteurs, et ce qui reste **ouvert**.

| | |
|---|---|
| **Série** | `InformationCalcul/Complexity`, accrétion de **03** (k) |
| **Public** | Recherche (fin de licence / master) |
| **Technologies** | C# 12 (.NET Interactive), bibliothèque standard seule ; les figures du carnet Python sont rendues ici en texte, un noyau C# n'ayant pas de matplotlib |
| **Durée** | 90 min |

## Objectifs

1. Compter les **comparaisons de poids** — la seule ressource que le modèle autorise — et rien d'autre.
2. Mesurer les trois classiques (Kruskal, Prim, Borůvka) sur des graphes de tailles croissantes, et voir **quel** facteur chacune paie.
3. Calculer **exactement** la complexité de décision `T*(H)` d'un petit graphe, et la distinguer de la complexité d'un **algorithme**.
4. Calculer **exactement** `alpha(m,n)` et situer la borne historique `m * alpha(m,n)` face à la borne linéaire.
5. Mesurer les deux certificats qui font vivre l'algorithme : le filtre **F-lourde / F-légère**, et le lemme d'**échantillonnage** de Karger-Klein-Tarjan.
6. Assembler un jouet qui a la **silhouette** de l'algorithme du papier, et mesurer où son coût part.

## Prérequis

Arbres couvrants et cycles (propriété du cycle), tri fusion et sa récurrence, notation asymptotique, logarithme itéré. Aucune connaissance préalable de la fonction d'Ackermann : elle est construite ici.

**Navigation** : [README de la série](README.md) | [Version Python](Complexity-03k-KhannaPuttermanSong-ArbreCouvrantLineaireDeterministe-Python.ipynb) | [Carnet 03 de base](Complexity-03-HierarchieTemporelle-Python.ipynb)

***

## Ce que ce carnet mesure — et ce qu'il ne fait pas

Le résultat du preprint tient en une ligne : la complexité en **comparaisons** de l'arbre couvrant minimal est **linéaire**, `T*(m,n) = O(m)`, là où l'on savait `T*(m,n) = O(m alpha(m,n))` et `Omega(m)`. Par une réduction antérieure (Pettie-Ramachandran, JACM 2002), cela donne un **algorithme** déterministe en `O(m)`.

Ce carnet prend le résultat comme une **boîte noire citée** et mesure les objets qui le rendent intelligible :

| Section | Objet mesuré | Ce que la mesure peut montrer |
|---|---|---|
| 2 | pire cas exact du tri fusion | l'unité de coût, et le `m log m` que Kruskal doit payer |
| 3 | Kruskal, Prim, Borůvka comptés | quel facteur logarithmique chacune porte |
| 4 | `T*(H)` exact | la différence **décision** / **algorithme** |
| 5 | `alpha(m,n)` exact | que `m * alpha` vaut au plus `4m` sur toute entrée réaliste |
| 6 | filtre F-lourde / F-légère | ce qu'on peut jeter sans perdre l'arbre |
| 7 | lemme d'échantillonnage | pourquoi le filtre aléatoire tient en espérance |
| 8 | jouet Borůvka + échantillon + filtre | où le coût part dans une architecture concrète |

**Ce que ce carnet ne fait pas**, et qu'aucune de ses sorties ne prouve : il ne démontre pas `T*(m,n) = O(m)`, il ne reproduit pas l'algorithme du preprint, il ne borne pas sa constante, et il ne dit rien de la validité du preprint, qui n'est pas évalué par les pairs.

Trois statuts sont employés partout dans la suite :

- **MESURÉ** — un nombre produit par une cellule de ce carnet, sur cette machine.
- **CITÉ** — une phrase du texte des auteurs, recopiée et attribuée.
- **OUVERT** — une question que rien ici ne tranche.

In [1]:
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Globalization;
using System.Linq;
using System.Numerics;
using System.Text;

CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentUICulture = CultureInfo.InvariantCulture;

int GRAINE = 20261126;
// Outillage commun : bibliotheque standard .NET uniquement, aucune dependance externe.
Console.WriteLine($"{System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription} ; BigInteger ; graine {GRAINE}");

.NET 10.0.15 ; BigInteger ; graine 20261126


## 1. Le modèle de coût : comparer, et rien d'autre

Le preprint se place dans le modèle le plus pauvre possible. **CITÉ** :

> We study the most general version of the problem, where the algorithm may only compare edge weights in pairs, with each comparison taking constant time, and otherwise treats the weights as opaque keys (thus, the algorithm works for even the most unstructured weighted settings).
>
> — TR26-339, §1

Conséquence opérationnelle : un poids n'est jamais lu, jamais additionné, jamais arrondi. La seule instruction autorisée sur deux poids est « le premier est-il strictement plus petit que le second ? ». Tout ce qui suit compte donc **une seule chose** : le nombre d'appels à cette comparaison.

C'est ce qui rend les poids entiers de `1` à `m` légitimes pour toutes les expériences de ce carnet : le modèle ne voit que leur **ordre**, et un tirage aléatoire de cet ordre est exactement une instance du problème. Les poids distincts évitent en plus les ex aequo, que la définition de `T*` exclut elle aussi (section 4).

L'instrument est une classe de trois lignes, `Comparaisons`, dont l'unique méthode `Inferieur(a, b)` incrémente un compteur et rend le verdict. Chaque algorithme reçoit une instance et ne touche aux poids qu'à travers elle : c'est la règle du jeu, pas une commodité de mesure.

## 2. L'instrument : un tri fusion dont le pire cas est exact

Pourquoi commencer par un tri ? Parce que les trois classiques de la section 3 en contiennent un, explicitement ou non : Kruskal trie ses arêtes, Prim maintient un tas, Borůvka trie implicitement à chaque phase. Le coût de comparaison d'un tri est donc l'étalon de tout le reste, et il vaut mieux le connaître exactement que de le majorer.

Le tri fusion descendant coupe une liste de `n` éléments en deux, trie chaque moitié, puis fusionne. Fusionner deux listes de tailles totales `n` demande au plus `n - 1` comparaisons, et ce maximum est atteint. D'où la récurrence, avec `C(0) = C(1) = 0` :

```
C(n) = C(floor(n/2)) + C(n - floor(n/2)) + (n - 1)
```

Sa solution est connue et close : avec `k = ceil(log2 n)`,

```
C(n) = n * k - 2**k + 1
```

La cellule suivante vérifie cette forme de trois façons indépendantes : par **force brute** sur toutes les permutations jusqu'à `n = 8`, par la **récurrence** jusqu'à `n = 400`, et par une **identité marginale** — le coût du `n`-ième élément vaut exactement `ceil(log2 n)` — jusqu'à `n = 2000`. Trois vérifications, parce qu'une seule forme close vérifiée sur ses propres valeurs ne prouve rien.

In [2]:
// L'instrument : un algorithme ne touche aux poids que par Inferieur(a, b).
class Comparaisons
{
    public long N;
    public bool Inferieur(double a, double b) { N += 1; return a < b; }
}

void Exiger(bool condition, string message) { if (!condition) throw new Exception(message); }

// Tri fusion descendant, instrumente.
int[] TriFusion(int[] items, double[] w, Comparaisons cmp)
{
    if (items.Length <= 1) return (int[])items.Clone();
    int milieu = items.Length / 2;
    int[] gauche = TriFusion(items[..milieu], w, cmp);
    int[] droite = TriFusion(items[milieu..], w, cmp);
    var fusion = new List<int>(items.Length);
    int i = 0, j = 0;
    while (i < gauche.Length && j < droite.Length)
    {
        if (cmp.Inferieur(w[gauche[i]], w[droite[j]])) { fusion.Add(gauche[i]); i += 1; }
        else { fusion.Add(droite[j]); j += 1; }
    }
    for (; i < gauche.Length; i += 1) fusion.Add(gauche[i]);
    for (; j < droite.Length; j += 1) fusion.Add(droite[j]);
    return fusion.ToArray();
}

int BitLength(int x) { int k = 0; while (x > 0) { k += 1; x >>= 1; } return k; }

// Pire cas exact : C(n) = C(n/2) + C(n - n/2) + n - 1, ferme par C(n) = n*k - 2^k + 1.
int FormeClose(int n)
{
    if (n <= 1) return 0;
    int k = BitLength(n - 1);
    return n * k - (1 << k) + 1;
}

List<int[]> ToutesPermutations(int n)
{
    var resultat = new List<int[]>();
    void Marcher(int[] courant, bool[] pris, int prof)
    {
        if (prof == n) { resultat.Add((int[])courant.Clone()); return; }
        for (int v = 0; v < n; v += 1)
        {
            if (pris[v]) continue;
            pris[v] = true;
            courant[prof] = v;
            Marcher(courant, pris, prof + 1);
            pris[v] = false;
        }
    }
    Marcher(new int[n], new bool[n], 0);
    return resultat;
}

Console.WriteLine("Pire cas du tri fusion, max sur toutes les permutations :");
for (int n = 1; n <= 8; n += 1)
{
    int pire = 0;
    foreach (int[] perm in ToutesPermutations(n))
    {
        var c = new Comparaisons();
        var w = new double[n];
        for (int i = 0; i < n; i += 1) w[i] = perm[i];
        TriFusion(Enumerable.Range(0, n).ToArray(), w, c);
        pire = Math.Max(pire, (int)c.N);
    }
    Exiger(pire == FormeClose(n), $"n={n} : max mesure {pire}, forme close {FormeClose(n)}");
    Console.WriteLine($"   n={n}  max mesure={pire,2}  forme close={FormeClose(n),2}");
}

for (int n = 9; n <= 400; n += 1)
    Exiger(FormeClose(n) == FormeClose(n / 2) + FormeClose(n - n / 2) + n - 1, $"recurrence n={n}");
Console.WriteLine("   recurrence <-> forme close verifiee sur n = 9..400");

for (int n = 1; n <= 2000; n += 1)
    Exiger(FormeClose(n) - FormeClose(n - 1) == BitLength(n - 1), $"cout marginal n={n}");
Console.WriteLine("   cout marginal C(n) - C(n-1) = ceil(log2 n), verifie sur n = 1..2000");
Console.WriteLine("   couts marginaux pour n = 1..16 : "
    + string.Join(" ", Enumerable.Range(1, 16).Select(n => FormeClose(n) - FormeClose(n - 1))));
Console.WriteLine();

Console.WriteLine("   {0,8} {1,9} {2,8} {3,8}", "n", "C(n)", "C(n)/n", "log2 n");
for (int k = 1; k <= 11; k += 1)
{
    int n = 1 << k;
    Console.WriteLine($"   {n,8} {FormeClose(n),9} {FormeClose(n) / (double)n,8:F3} {k,8}");
}
Console.WriteLine($"   C(1000) = {FormeClose(1000)} : pire cas exact du tri de 1000 elements.");

Pire cas du tri fusion, max sur toutes les permutations :


   n=1  max mesure= 0  forme close= 0


   n=2  max mesure= 1  forme close= 1


   n=3  max mesure= 3  forme close= 3


   n=4  max mesure= 5  forme close= 5


   n=5  max mesure= 8  forme close= 8


... (15 outputs left out)

        512      4097    8.002        9


       1024      9217    9.001       10


       2048     20481   10.000       11


   C(1000) = 8977 : pire cas exact du tri de 1000 elements.


### Lecture du résultat

Les trois vérifications passent, et elles disent trois choses différentes.

**La force brute** (lignes `n=1..8`) compare le maximum mesuré sur *toutes* les permutations à la forme close : `0, 1, 3, 5, 8, 11, 14, 17`. C'est une égalité exhaustive jusqu'à `n = 8`, pas un échantillon.

**La récurrence** (ligne `9..400`) montre que la forme close satisfait bien la récurrence qu'elle prétend résoudre — utile, parce qu'une erreur d'arrondi sur `n` impair se verrait là et nulle part ailleurs.

**L'identité marginale** est la plus parlante : `C(n) - C(n-1) = ceil(log2 n)`, vérifié sur `n = 1..2000`. Ajouter le `n`-ième élément coûte exactement `ceil(log2 n)` comparaisons, quels que soient les poids. La suite des coûts marginaux pour `n = 1..16` le montre à l'œil :

```
0  1  2  2  3  3  3  3  4  4  4  4  4  4  4  4
```

Chaque palier dure deux fois plus longtemps que le précédent : c'est la signature d'un logarithme. Et ce logarithme n'est pas une maladresse du tri fusion — aucun tri par comparaison ne peut faire mieux, dans le pire cas, que `ceil(log2(n!))` comparaisons (un arbre de décision binaire de profondeur `d` a au plus `2**d` feuilles, et il faut une feuille par permutation).

Pour `n = 2**k`, la table donne `C(n)/n = k - 1 + 1/n`, c'est-à-dire

```
C(n) = n * log2(n) - n + 1          (n puissance de deux)
```

soit exactement **une comparaison par élément de moins** que la borne `n log2 n` qu'on cite souvent. Pour `n = 1000`, la forme close donne `C(1000) = 8977`, à comparer aux `9966` de `n log2 n`.

**MESURÉ, et c'est le point à retenir pour la section 7** : `8977` est un maximum sur les permutations, pas le coût d'une entrée particulière. Un tri donné peut coûter beaucoup moins — la section 7 en mesurera deux, à `8694` et `4932` comparaisons pour 1000 poids. Mais un algorithme déterministe qui doit trier **quelle que soit l'entrée** paie `Theta(m log m)` dans le pire cas : c'est précisément ce que la borne linéaire du preprint doit éviter de payer.

La cellule suivante met les deux suites côte à côte, en texte : le coût marginal et `ceil(log2 n)`, une barre par unité.

In [3]:
int n_max = 40;
Console.WriteLine("Tri fusion : le cout marginal vaut exactement ceil(log2 n).");
Console.WriteLine("   {0,4} {1,10} {2,12} {3,9}  {4}", "n", "marginal", "ceil(log2 n)", "log2(n)", "barre");
for (int n = 1; n <= n_max; n += 1)
{
    int marginal = FormeClose(n) - FormeClose(n - 1);
    double lg = Math.Log2(n);
    string barre = new string('#', (int)Math.Ceiling(lg));
    Console.WriteLine($"   {n,4} {marginal,10} {(int)Math.Ceiling(lg),12} {lg,9:F3}  {barre}");
}
Console.WriteLine();
Console.WriteLine("   les paliers, la ou la marche monte :");
for (int k = 1; k <= 6; k += 1)
{
    int bas = (1 << (k - 1)) + 1, haut = 1 << k;
    Console.WriteLine($"     k={k} : n de {bas,2} a {haut,2} -> cout marginal {k}");
}
Console.WriteLine("   au-dela de n = 32 le palier k=6 court jusqu'a n = 64 : le cout marginal");
Console.WriteLine("   ne monte qu'aux puissances de 2, et d'exactement 1 a chaque fois.");

Tri fusion : le cout marginal vaut exactement ceil(log2 n).


      n   marginal ceil(log2 n)   log2(n)  barre


      1          0            0     0.000  


      2          1            1     1.000  #


      3          2            2     1.585  ##


      4          2            2     2.000  ##


... (41 outputs left out)

     k=5 : n de 17 a 32 -> cout marginal 5


     k=6 : n de 33 a 64 -> cout marginal 6


   au-dela de n = 32 le palier k=6 court jusqu'a n = 64 : le cout marginal


   ne monte qu'aux puissances de 2, et d'exactement 1 a chaque fois.


### Lecture du tableau

Les deux colonnes se superposent ligne à ligne : le coût marginal mesuré et `ceil(log2 n)` coïncident, et la barre a exactement la longueur du logarithme arrondi au plafond. La colonne `log2(n)` en valeur réelle montre l'écart que le plafond comble : entre deux puissances de deux, elle glisse alors que le coût, lui, est en marche d'escalier.

La marche est large d'un facteur deux, ce qui est exactement la structure d'un arbre de fusion — chaque niveau de l'arbre double le nombre de sous-listes, donc la profondeur augmente d'une unité tous les doubles. Les paliers listés en fin de sortie le disent sans détour : `n` de 33 à 64 vaut 6, comme `n` de 17 à 32 valait 5.

Trois conséquences pour la suite :

1. Un tri coûte `n log2 n` **au niveau de l'élément**, mais chaque élément ne paie le logarithme qu'autant qu'il participe à des fusions de listes de plus en plus grandes.
2. Le coût est **déterministe** : il ne dépend que de `n`, jamais des valeurs. C'est pour cela que `C(n)` peut servir d'étalon.
3. Le pire cas est atteint, mais pas par une entrée privilégiée. C'est une différence de nature avec la borne `T*(H)` de la section 4, qui est un **minimum** sur les arbres de décision — donc une borne atteignable par un algorithme bien choisi.

La section suivante met ce tri au travail : Kruskal en contient un, et c'est là que le facteur `log` entre dans le coût des classiques.

## 3. Trois classiques, comptés

Les trois algorithmes historiques sont tous **gloutons** et tous corrects : ils rendent le même arbre. Ce qui les distingue n'est donc pas le résultat, mais **où partent les comparaisons**. Le preprint les décrit lui-même, **CITÉ** (§1.1) :

> In 1930, Jarník described a tree-growing alternative. Indeed, this work proposed starting from a single vertex and repeatedly adding the cheapest edge leaving the current tree. This algorithm was later rediscovered by Prim and by Dijkstra and is now commonly called Prim's algorithm. Kruskal developed yet another alternative. In his work, he proposes scanning the edges in increasing order of weight and keeping each edge that closes no cycle.

et pour Borůvka :

> each vertex selects its incident edge with the smallest weight. The algorithm then commits to keeping these edges in the MST, and effectively contracts on these committed edges, treating the created components as their own vertices. In this way, every phase at least halves the number of vertices, so after O(log n) phases, each using O(m) comparisons, one obtains an MST.

Le preprint résume aussi le coût de chacun, **CITÉ** (§1.1) :

> Kruskal's algorithm sorts the edges in O(m log m) = O(m log n) time, Prim's algorithm performs O(m) operations on a binary heap of at most n vertices, and Borůvka's algorithm may need Θ(log n) linear-time phases.

Trois implémentations suivent, toutes écrites à la main et instrumentées de la même façon :

- **Kruskal** : tri de toutes les arêtes, puis un Union-Find qui accepte une arête si elle relie deux composantes distinctes.
- **Prim** : un tas binaire de **clés = arêtes**, alimenté par les voisinages ; on retire le minimum, on l'ajoute si son extrémité est neuve.
- **Borůvka** : à chaque phase, chaque composante retient son arête sortante la plus légère ; les composantes fusionnent, on recommence. Les arêtes internes sont retirées avant la phase suivante.

Le Union-Find compte aussi ses propres pas de boucle (`appels`, `pas`), ce qui servira en section 5. Les poids sont des entiers distincts tirés au hasard : le graphe ne sait rien de son ordre, et le modèle de comparaison ne voit que cet ordre.

La cellule de mesure vérifie que les trois rendent le même arbre : par `Exiger`, la liste triée des poids de Prim et celle de Borůvka sont égales à celle de Kruskal. Cela entraîne l'égalité des poids totaux ; comme Kruskal rend un arbre minimal, les deux autres aussi, et des poids distincts le rendent unique.

In [4]:
// Union-Find, compression de chemin par fractionnement ; compte ses pas de boucle.
class UnionFind
{
    public int[] Parent;
    public long Appels;
    public long Pas;

    public UnionFind(int n)
    {
        Parent = new int[n];
        for (int i = 0; i < n; i += 1) Parent[i] = i;
    }

    public UnionFind Copie()
    {
        var u = new UnionFind(0);
        u.Parent = (int[])Parent.Clone();
        return u;
    }

    public int Trouver(int x)
    {
        int[] p = Parent;
        Appels += 1;
        while (p[x] != x)
        {
            p[x] = p[p[x]];
            x = p[x];
            Pas += 1;
        }
        return x;
    }

    public bool Unir(int a, int b)
    {
        int ra = Trouver(a), rb = Trouver(b);
        if (ra == rb) return false;
        Parent[ra] = rb;
        return true;
    }
}

// n sommets, environ ratio*n aretes, poids distincts melanges : le graphe ne sait rien de son ordre.
((int U, int V)[] Aretes, double[] W) GrapheAleatoire(int n, double ratio, int graine)
{
    var rng = new Random(graine);
    var aretes = new List<(int U, int V)>();
    var vues = new HashSet<(int, int)>();
    for (int v = 1; v < n; v += 1)
    {
        int u = rng.Next(v);
        aretes.Add((u, v));
        vues.Add((u, v));
    }
    int objectif = Math.Min((int)(ratio * n), n * (n - 1) / 2);
    while (aretes.Count < objectif)
    {
        int u = rng.Next(n), v = rng.Next(n);
        if (u == v) continue;
        var cle = (Math.Min(u, v), Math.Max(u, v));
        if (!vues.Add(cle)) continue;
        aretes.Add(cle);
    }
    var w = new double[aretes.Count];
    var ordre = Enumerable.Range(0, aretes.Count).ToArray();
    for (int i = ordre.Length - 1; i > 0; i -= 1)
    {
        int j = rng.Next(i + 1);
        (ordre[i], ordre[j]) = (ordre[j], ordre[i]);
    }
    for (int k = 0; k < ordre.Length; k += 1) w[ordre[k]] = k + 1;
    return (aretes.ToArray(), w);
}

List<int> Kruskal(int[] E, (int U, int V)[] aretes, double[] w, Comparaisons cmp, UnionFind uf)
{
    var arbre = new List<int>();
    foreach (int e in TriFusion(E, w, cmp))
        if (uf.Unir(aretes[e].U, aretes[e].V)) arbre.Add(e);
    return arbre;
}

// Prim, tas binaire ecrit a la main ; les cles sont les aretes (poids distincts : 1 comparaison).
List<int> Prim(int n, (int U, int V)[] aretes, double[] w, Comparaisons cmp)
{
    var adj = new List<(int E, int V)>[n];
    for (int i = 0; i < n; i += 1) adj[i] = new List<(int E, int V)>();
    for (int e = 0; e < aretes.Length; e += 1)
    {
        adj[aretes[e].U].Add((e, aretes[e].V));
        adj[aretes[e].V].Add((e, aretes[e].U));
    }
    var tas = new List<int>();

    bool Moins(int e1, int e2) => cmp.Inferieur(w[e1], w[e2]);

    void Pousser(int e)
    {
        tas.Add(e);
        int i = tas.Count - 1;
        while (i > 0)
        {
            int p = (i - 1) / 2;
            if (Moins(tas[i], tas[p])) { (tas[i], tas[p]) = (tas[p], tas[i]); i = p; }
            else break;
        }
    }

    int Retirer()
    {
        int tete = tas[0];
        int dernier = tas[tas.Count - 1];
        tas.RemoveAt(tas.Count - 1);
        if (tas.Count > 0)
        {
            tas[0] = dernier;
            int i = 0;
            while (true)
            {
                int g = 2 * i + 1, d = 2 * i + 2, s = i;
                if (g < tas.Count && Moins(tas[g], tas[s])) s = g;
                if (d < tas.Count && Moins(tas[d], tas[s])) s = d;
                if (s == i) break;
                (tas[i], tas[s]) = (tas[s], tas[i]);
                i = s;
            }
        }
        return tete;
    }

    var vu = new bool[n];
    var arbre = new List<int>();
    vu[0] = true;
    foreach (var (e, _) in adj[0]) Pousser(e);
    while (tas.Count > 0)
    {
        int e = Retirer();
        if (vu[aretes[e].U] && vu[aretes[e].V]) continue;
        arbre.Add(e);
        int x = vu[aretes[e].U] ? aretes[e].V : aretes[e].U;
        vu[x] = true;
        foreach (var (f, y) in adj[x])
            if (!vu[y]) Pousser(f);
    }
    return arbre;
}

// Chaque composante garde son arete sortante la plus legere : elle est dans l'arbre.
List<int> PhaseBoruvka(int[] E, (int U, int V)[] aretes, double[] w, Comparaisons cmp, UnionFind uf)
{
    var meilleure = new Dictionary<int, int>();
    foreach (int e in E)
    {
        int ru = uf.Trouver(aretes[e].U), rv = uf.Trouver(aretes[e].V);
        if (ru == rv) continue;
        foreach (int r in new[] { ru, rv })
        {
            if (!meilleure.TryGetValue(r, out int courant)) meilleure[r] = e;
            else if (cmp.Inferieur(w[e], w[courant])) meilleure[r] = e;
        }
    }
    var ajoutees = new List<int>();
    foreach (int e in meilleure.Values)
        if (uf.Unir(aretes[e].U, aretes[e].V)) ajoutees.Add(e);
    return ajoutees;
}

// (aretes de l'arbre, nombre de phases) : chaque phase au moins divise les composantes par 2.
(List<int> Arbre, int Phases) Boruvka(int[] E, (int U, int V)[] aretes, double[] w, Comparaisons cmp, UnionFind uf)
{
    var arbre = new List<int>();
    int phases = 0;
    while (true)
    {
        var ajoutees = PhaseBoruvka(E, aretes, w, cmp, uf);
        if (ajoutees.Count == 0) return (arbre, phases);
        phases += 1;
        arbre.AddRange(ajoutees);
        E = E.Where(e => uf.Trouver(aretes[e].U) != uf.Trouver(aretes[e].V)).ToArray();
    }
}

Console.WriteLine("Outils en place : Union-Find, generateur de graphes, Kruskal, Prim, Boruvka.");

Outils en place : Union-Find, generateur de graphes, Kruskal, Prim, Boruvka.


Les trois algorithmes et le générateur de graphes sont en place. La cellule suivante les fait tourner sur six tailles, de 500 à 16 000 sommets, à densité fixée `m = 3n`, et rapporte les comparaisons **par arête** — une unité qui rend les tailles comparables.

In [5]:
bool MemesPoids(List<int> a, List<int> b, double[] w)
{
    var x = a.Select(e => w[e]).OrderBy(v => v).ToArray();
    var y = b.Select(e => w[e]).OrderBy(v => v).ToArray();
    if (x.Length != y.Length) return false;
    for (int i = 0; i < x.Length; i += 1) if (x[i] != y[i]) return false;
    return true;
}

Console.WriteLine("Comparaisons de poids par arete, densite m = 3n :");
Console.WriteLine("   {0,6} {1,7} {2,8} {3,8} {4,8} {5,7} {6,7} {7,7}",
                  "n", "m", "Kruskal", "Prim", "Boruvka", "phases", "log2 n", "s");
var lignes = new List<(int N, int M, double K, double P, double B, int Phases)>();
foreach (int n in new[] { 500, 1000, 2000, 4000, 8000, 16000 })
{
    var (aretes, w) = GrapheAleatoire(n, 3.0, n);
    int m = aretes.Length;
    var montre = Stopwatch.StartNew();

    var cmpK = new Comparaisons();
    var arbreK = Kruskal(Enumerable.Range(0, m).ToArray(), aretes, w, cmpK, new UnionFind(n));

    var cmpP = new Comparaisons();
    var arbreP = Prim(n, aretes, w, cmpP);
    Exiger(MemesPoids(arbreP, arbreK, w), $"Prim ne rend pas l'arbre de Kruskal (n={n})");

    var cmpB = new Comparaisons();
    var (arbreB, phasesB) = Boruvka(Enumerable.Range(0, m).ToArray(), aretes, w, cmpB, new UnionFind(n));
    Exiger(MemesPoids(arbreB, arbreK, w), $"Boruvka ne rend pas l'arbre de Kruskal (n={n})");

    montre.Stop();
    lignes.Add((n, m, cmpK.N / (double)m, cmpP.N / (double)m, cmpB.N / (double)m, phasesB));
    Console.WriteLine($"   {n,6} {m,7} {cmpK.N / (double)m,8:F2} {cmpP.N / (double)m,8:F2}"
        + $" {cmpB.N / (double)m,8:F2} {phasesB,7} {Math.Log2(n),7:F1} {montre.Elapsed.TotalSeconds,7:F1}");
}
Console.WriteLine();
Console.WriteLine("   gain par doublement de n :");
for (int i = 1; i < lignes.Count; i += 1)
{
    var (prec, cour) = (lignes[i - 1], lignes[i]);
    Console.WriteLine($"   n {prec.N,6} -> {cour.N,6} : Kruskal {cour.K - prec.K:+0.00;-0.00}"
        + $"   Prim {cour.P - prec.P:+0.00;-0.00}   Boruvka {cour.B - prec.B:+0.00;-0.00}");
}
Console.WriteLine("   controle : les trois methodes rendent le meme arbre (memes poids, verifie par Exiger).");

Comparaisons de poids par arete, densite m = 3n :


        n       m  Kruskal     Prim  Boruvka  phases  log2 n       s


      500    1500     9.32    18.74     5.38       4     9.0     0.0


     1000    3000    10.31    21.05     5.51       4    10.0     0.0


     2000    6000    11.30    23.25     6.58       5    11.0     0.0


     4000   12000    12.31    25.50     7.43       6    12.0     0.0


... (5 outputs left out)

   n   2000 ->   4000 : Kruskal +1.00   Prim +2.26   Boruvka +0.86


   n   4000 ->   8000 : Kruskal +1.00   Prim +2.18   Boruvka +0.03


   n   8000 ->  16000 : Kruskal +1.00   Prim +2.16   Boruvka +0.22


   controle : les trois methodes rendent le meme arbre (memes poids, verifie par Exiger).


### Lecture du résultat

Le contrôle `Exiger` passe : Prim et Borůvka rendent l'arbre de Kruskal sur les six tailles. Les colonnes se lisent en **comparaisons par arête**, donc le total s'obtient en multipliant par `m`.

**Kruskal : `+1,00` par doublement.** C'est le coût du tri, et rien d'autre : le Union-Find ne fait aucune comparaison de poids. Les gains `+0,99, +0,99, +1,00, +1,00, +1,00` sont exactement le `+1` que prend `log2 n` quand `n` double, et la colonne `K-log2 n` de la seconde table le confirme : Kruskal vaut `log2 n` plus une constante de l'ordre de `0,3`. Quant au niveau (`9,32` à `14,30` par arête), il reste **sous** le pire cas exact de la section 2 : `C(1500) = 14453` et `C(48000) = 702465`, soit `9,64` et `14,63` par arête. La permutation aléatoire n'atteint pas le pire cas, on l'avait annoncé.

**Prim : environ `+2,2` par doublement.** Chaque arête entre une fois dans le tas et en sort une fois, et chaque opération au tas parcourt un chemin de profondeur logarithmique en comparant jusqu'à deux enfants par niveau. D'où un coefficient voisin de `2` devant `log2 n` : `18,74` pour `log2 n = 9`, `29,84` pour `log2 n = 14`. Prim est le plus coûteux des trois **en comparaisons**, sur des graphes de densité 3 — ce qui ne dit rien de son temps de calcul réel, que ce carnet ne mesure pas.

**Borůvka : le plus bas, et le plus plat.** De `5,38` à `7,69` comparaisons par arête, pour des gains de `+0,13` à `+1,06` par doublement, sans monotonie : la croissance est bien plus lente que logarithmique sur cette plage. La colonne `phases` explique pourquoi : il y a `4` à `6` phases, alors que `log2 n` vaut `9` à `14`. La borne « every phase at least halves the number of vertices » du preprint est une borne **supérieure** ; la trace de la section 8 montrera combien de composantes une seule phase fait réellement tomber. En moyenne, une phase coûte ici entre `1,24` et `1,38` comparaison par arête : c'est le rapport des colonnes `Boruvka` et `phases`, ligne par ligne (`5,38 / 4 = 1,35` à `n = 500`, `7,43 / 6 = 1,24` à `n = 4000`).

**Ce que la table ne prouve pas.** Borůvka est le moins coûteux sur ces six tailles, mais son coût **croît** (`5,38 → 7,69`) et le preprint le donne `O(m log n)` au pire : la ligne ne suffit pas à distinguer `m log log n` de `m alpha(m,n)`, ni l'un ni l'autre de `m`. Trancher demanderait des tailles inaccessibles ici ; c'est tout l'objet des sections 4 et 5.

La colonne `s` (temps en secondes) n'est qu'indicative : elle varie d'une machine à l'autre, alors que les comparaisons, elles, sont reproductibles au bit près. Les valeurs de cette table diffèrent légèrement de celles du carnet Python — les générateurs pseudo-aléatoires des deux langages ne tirent pas les mêmes graphes — mais les **pentes** par doublement, elles, coïncident, et c'est elles que les sections suivantes exploitent.

In [6]:
Console.WriteLine("Trois facons de trouver le meme arbre : comparaisons de poids par arete.");
Console.WriteLine("1 caractere = 1 comparaison par arete (densite m = 3n).");
Console.WriteLine("   {0,6} {1,7} {2,8} {3,8} {4,8} {5,9}", "n", "log2 n", "Kruskal", "Prim", "Boruvka", "K-log2 n");
foreach (var l in lignes)
{
    double ecart = l.K - Math.Log2(l.N);
    Console.WriteLine($"   {l.N,6} {Math.Log2(l.N),7:F1} {l.K,8:F2} {l.P,8:F2} {l.B,8:F2} {ecart,9:F2}");
    Console.WriteLine($"      Kruskal {new string('#', (int)Math.Round(l.K))}");
    Console.WriteLine($"      Prim    {new string('#', (int)Math.Round(l.P))}");
    Console.WriteLine($"      Boruvka {new string('#', (int)Math.Round(l.B))}");
}
Console.WriteLine();
Console.WriteLine("   pente par doublement de n (comparaisons par arete) :");
for (int i = 1; i < lignes.Count; i += 1)
{
    var (prec, cour) = (lignes[i - 1], lignes[i]);
    Console.WriteLine($"     n {prec.N,6} -> {cour.N,6} : Kruskal {cour.K - prec.K:+0.00;-0.00}"
        + $"   Prim {cour.P - prec.P:+0.00;-0.00}   Boruvka {cour.B - prec.B:+0.00;-0.00}");
}

Trois facons de trouver le meme arbre : comparaisons de poids par arete.


1 caractere = 1 comparaison par arete (densite m = 3n).


        n  log2 n  Kruskal     Prim  Boruvka  K-log2 n


      500     9.0     9.32    18.74     5.38      0.35


      Kruskal #########


      Prim    ###################


... (23 outputs left out)

     n   1000 ->   2000 : Kruskal +0.99   Prim +2.20   Boruvka +1.06


     n   2000 ->   4000 : Kruskal +1.00   Prim +2.26   Boruvka +0.86


     n   4000 ->   8000 : Kruskal +1.00   Prim +2.18   Boruvka +0.03


     n   8000 ->  16000 : Kruskal +1.00   Prim +2.16   Boruvka +0.22


### Lecture du tableau

La colonne `log2 n` sert d'échelle : si le coût par arête est de la forme `s * log2 n + c`, alors le rapport `coût / log2 n` **tend vers `s`**, et la colonne `K - log2 n` doit rester **constante** quand `n` grandit. C'est exactement ce que la table montre pour Kruskal.

- **Kruskal** : `K - log2 n` vaut `0,35` puis `0,34` sur les cinq doublements suivants, et les gains par doublement sont `+0,99`, `+0,99`, `+1,00`, `+1,00`, `+1,00`. La pente est `1` : c'est `m log m`, le prix du tri.
- **Prim** : les gains par doublement valent `+2,31`, `+2,20`, `+2,26`, `+2,18`, `+2,16`, soit une pente voisine de `2,2` : environ deux opérations au tas par arête, chacune logarithmique.
- **Borůvka** : les gains valent `+0,13`, `+1,06`, `+0,86`, `+0,03`, `+0,22`. La suite n'est pas monotone et **diminue** globalement : la courbe s'aplatit à mesure que `n` grandit. C'est le comportement attendu d'un algorithme dont les phases se raccourcissent quand le graphe grandit, mais rien ici ne démontre qu'elle resterait plate indéfiniment.

L'écart entre les trois colonnes est l'espace où se joue la question du preprint : pour passer de `O(m log n)` à `O(m)`, il faut gagner un facteur qui n'est pas visible sur une plage de cinq doublements. C'est pourquoi la suite change d'échelle et d'objet : la section 4 regarde des graphes de **six arêtes** et calcule une quantité exacte, plutôt que d'extrapoler une courbe.

Une remarque sur les durées : elles ne figurent ici que pour mémoire. Les nombres de comparaisons sont reproductibles (les graphes sont tirés par une graine fixée, et `Exiger` vérifie que les trois algorithmes rendent la même forêt), les temps d'horloge ne le sont pas.

## 4. La complexité de décision : T*(H)

Les trois algorithmes de la section 3 sont des programmes. Le preprint, lui, commence par une question plus petite et plus précise : **combien de comparaisons faut-il, au minimum, pour trouver l'arbre ?** Le texte des auteurs pose l'objet ainsi :

> « let us fix a simple undirected graph H. A comparison tree for H is a decision tree where each internal node compares two edge weights and whose leaves are labeled with spanning forests of H. We say that it is correct if, for every possible assignment of distinct weights to H, the leaf reached is labeled with the resulting MSF. The depth of the comparison tree is then exactly the worst-case number of comparisons required to find an MSF of H. With this, we can define »

puis la définition elle-même, recomposée : dans le PDF l'astérisque est un exposant, et `H` et `T` sont des indices placés sous `max` et sous `min`, que l'extraction du texte déplace en fin de ligne.

`T*(m, n) = max_H min_T depth(T)`

> « where H ranges over all simple undirected graphs with m edges and n vertices, including disconnected ones, and T ranges over all correct comparison trees for H. »

Deux précisions du texte méritent d'être gardées :

- **Pourquoi les graphes non connexes entrent dans le maximum.** Une note de bas de page des auteurs l'explique : un graphe connexe à `m = n - 1` arêtes est son propre arbre couvrant, donc `T*(n-1, n) = 0` si on ne regardait que les graphes connexes, alors que **lire** une telle entrée coûte déjà `Omega(n)`. La définition inclut les graphes non connexes pour que le coût de lecture ne disparaisse pas du problème.
- **Décision d'abord, algorithme ensuite.** Le preprint borne `T*(m, n)`, c'est-à-dire le **nombre de comparaisons**, pas directement le temps d'exécution. Le passage de l'un à l'autre est un théorème antérieur, que les auteurs citent explicitement : « Pettie and Ramachandran gave a uniform pointer-machine algorithm that matches this bound. Indeed, on connected inputs, their algorithm runs in `O(T ∗(m, n))` time ». C'est ce que veut dire la phrase du résumé : borner la complexité de comparaison, puis « a known reduction of Pettie and Ramachandran (JACM 2002) immediately implies a linear time algorithm ». Un arbre de décision optimal n'est pas un programme ; c'est la réduction qui en fabrique un.

### Ce que la cellule suivante calcule exactement

`T*(m, n)` est un maximum sur **toutes** les topologies à `m` arêtes et `n` sommets : hors d'atteinte par le calcul direct. Ce qui est calculable exactement, c'est le **terme intérieur** : pour une topologie `H` fixée, la profondeur du meilleur arbre de comparaison, par minimax mémoïsé.

- Un état est un **ordre partiel** sur les arêtes, représenté par un ensemble de relations « `a` est plus légère que `b` » et sa clôture transitive.
- Un nœud interne choisit une paire d'arêtes non encore comparées, et le coût est `1 + max(les deux branches)`.
- Une feuille est atteinte quand **toutes** les permutations encore compatibles avec l'ordre partiel donnent le **même** arbre couvrant minimal : plus aucune comparaison ne peut rien changer.

Le calcul porte sur neuf topologies de six arêtes au plus, dont les permutations se laissent énumérer (`6! = 720` au plus). C'est une mesure exacte sur de très petits graphes ; ce n'est pas `T*(m, n)`.

### Ce que la clôture a le droit d'inférer

Chaque réponse est une relation : « `a` est plus légère que `b` ». La clôture a le droit d'en déduire d'autres, mais seulement celles qui sont vraies dans **tout** ordre encore compatible avec ce qui est su. La seule règle fondée est celle-ci :

`x ≤ a < b ≤ y  ⟹  x < y`

Elle ne demande donc que le **demi-ensemble inférieur** de `a` (ce qui est connu plus léger que `a`, `a` compris) et le **demi-ensemble supérieur** de `b` (`b` compris, avec ce qui est connu plus lourd). Y glisser autre chose — les arêtes plus légères que `b`, ou les arêtes plus lourdes que `a` — fabrique des contraintes que rien ne justifie.

Le préjudice d'une telle invention ne se voit pas dans la table : un état trop contraint atteint plus tôt le moment où tous les ordres restants donnent le même arbre, donc la feuille se déclenche trop tôt et `T*` est **sous-estimé**. Le cas le plus court le montre. L'état ne contient qu'une relation, `0 < 1` ; on demande si `0 < 2`, et la réponse est oui. Cela n'apprend rien sur `1` et `2` : l'ordre `0 < 2 < 1` satisfait la relation connue et la réponse reçue. Une clôture qui en conclut `1 < 2`, au motif que `1` est plus lourd que `0`, a inventé une contrainte.

### Un second calcul, sans aucune inférence

Comme une erreur de clôture est silencieuse, la cellule suivante fait le travail deux fois. Le premier calcul est celui qui vient d'être décrit : ordre partiel, clôture, mémoïsation par état. Le second n'utilise **aucune inférence** : un état y est l'**ensemble des permutations encore compatibles**, une réponse coupe cet ensemble en deux, et le minimax se fait sur cette partition. Il est plus lent, mais il ne partage avec le premier que la définition de `T*` : si les deux tombent d'accord sur les neuf topologies, la clôture n'a rien fabriqué. Les deux valeurs sont imprimées côte à côte, et les états mémoïsés sont comptés dans les deux cas.

Deux détails d'implémentation en C# valent d'être signalés. L'ordre partiel est un tableau de masques d'entiers (`lt[y]` = ensemble des arêtes connues plus légères que `y`), et il est empaqueté dans un `long` — six bits par entrée, soit 36 bits pour six arêtes — pour servir de clé au dictionnaire de mémoïsation. La forêt, elle, est identifiée par une **chaîne** (`"0,2,3"`) et non par un tableau : en C#, deux tableaux de même contenu ne sont pas égaux, donc un `Distinct()` sur des `int[]` compterait des forêts identiques comme distinctes, et le test de feuille ne se déclencherait jamais.

In [7]:
// La MSF d'un ordre total sur les aretes, rendue comme cle comparable (indices tries, joints).
// Cle sur une chaine et non sur un tableau : un tableau compare par reference, pas par contenu.
string MsfDUnOrdre((int U, int V)[] aretes, int[] ordre, int n)
{
    var uf = new UnionFind(n);
    var foret = new List<int>();
    foreach (int e in ordre)
    {
        if (uf.Unir(aretes[e].U, aretes[e].V)) foret.Add(e);
    }
    foret.Sort();
    return string.Join(",", foret);
}

long Factorielle(int x)
{
    long r = 1;
    for (int i = 2; i <= x; i += 1) r *= i;
    return r;
}

// T*(H) : profondeur de l'arbre de comparaison optimal, par minimax memoise sur les ordres.
// A chaque noeud on essaie toutes les paires encore libres ; une comparaison est licite si elle
// ne contredit pas l'ordre partiel, et l'adversaire choisit la branche la plus profonde.
(int Valeur, int Etats, int NMsf, double Secondes) TEtoile((int U, int V)[] aretes, int n)
{
    int m = aretes.Length;
    var ordres = ToutesPermutations(m);
    var rang = ordres.Select(o =>
    {
        var d = new Dictionary<int, int>();
        for (int k = 0; k < o.Length; k += 1) d[o[k]] = k;
        return d;
    }).ToList();
    var msf = ordres.Select(o => MsfDUnOrdre(aretes, o, n)).ToList();
    var memo = new Dictionary<long, int>();

    // L'ordre partiel est un tableau de masques : lt[y] = aretes connues plus legeres que y.
    // Six bits par entree suffisent (m <= 6 ici), d'ou une cle long pour la memoisation.
    long Coder(int[] lt)
    {
        long v = 0;
        for (int i = 0; i < lt.Length; i += 1) v |= ((long)lt[i]) << (6 * i);
        return v;
    }

    // Point fixe de transitivite : si x est plus leger que y, tout ce qui est plus leger que x
    // est plus leger que y. La fermeture est complete avant le test de feuille, sinon deux
    // etats differents se confondraient sous la meme cle.
    int[] PointFixe(int[] lt)
    {
        var ferme = (int[])lt.Clone();
        bool change = true;
        while (change)
        {
            change = false;
            for (int y = 0; y < m; y += 1)
            {
                int acc = ferme[y];
                int bits = acc;
                while (bits != 0)
                {
                    int x = BitOperations.TrailingZeroCount(bits);
                    bits &= bits - 1;
                    acc |= ferme[x];
                }
                if (acc != ferme[y])
                {
                    ferme[y] = acc;
                    change = true;
                }
            }
        }
        return ferme;
    }

    // Ajoute « a plus legere que b ». La seule inference fondee est x <= a < b <= y => x < y :
    // elle demande le demi-ensemble INFERIEUR de a (lt[a], plus a) et le demi-ensemble
    // SUPERIEUR de b (b, plus les y tels que b < y). Y glisser les predecesseurs de b ou les
    // successeurs de a invente des contraintes : sur l'etat {0 < 1}, la reponse a « 0 < 2 ? »
    // n'apprend rien sur 1 et 2, et l'ordre 0 < 2 < 1 le montre.
    (int[] Lt, List<(int X, int Y)> Paires)? Ajouter(int[] lt, int a, int b)
    {
        if (((lt[a] >> b) & 1) != 0 || ((lt[b] >> a) & 1) != 0) return null;
        int pred = lt[a] | (1 << a);
        int succ = 1 << b;
        for (int y = 0; y < m; y += 1)
        {
            if (((lt[y] >> b) & 1) != 0) succ |= 1 << y;
        }
        var brut = (int[])lt.Clone();
        for (int y = 0; y < m; y += 1)
        {
            if (((succ >> y) & 1) != 0) brut[y] |= pred;
        }
        var lt2 = PointFixe(brut);
        var nouvelles = new List<(int X, int Y)>();
        for (int y = 0; y < m; y += 1)
        {
            for (int x = 0; x < m; x += 1)
            {
                if (((lt2[y] >> x) & 1) != 0 && ((lt[y] >> x) & 1) == 0) nouvelles.Add((x, y));
            }
        }
        return (lt2, nouvelles);
    }

    int Valeur(int[] lt, List<int> extensions)
    {
        long cle = Coder(lt);
        if (memo.TryGetValue(cle, out int cache)) return cache;
        // Feuille : tous les ordres encore compatibles rendent la meme foret.
        if (extensions.Select(i => msf[i]).Distinct().Count() == 1)
        {
            memo[cle] = 0;
            return 0;
        }
        int meilleure = 1000000000;
        for (int a = 0; a < m; a += 1)
        {
            for (int b = a + 1; b < m; b += 1)
            {
                if (((lt[a] >> b) & 1) != 0 || ((lt[b] >> a) & 1) != 0) continue;
                var r1 = Ajouter(lt, a, b);
                if (r1 is null) continue;
                var (lt1, paires1) = r1.Value;
                var ext1 = extensions
                    .Where(i => paires1.All(p => rang[i][p.X] < rang[i][p.Y]))
                    .ToList();
                int v1 = Valeur(lt1, ext1);
                if (1 + v1 >= meilleure) continue;
                var r2 = Ajouter(lt, b, a);
                if (r2 is null) continue;
                var (lt2, paires2) = r2.Value;
                var ext2 = extensions
                    .Where(i => paires2.All(p => rang[i][p.X] < rang[i][p.Y]))
                    .ToList();
                int v2 = Valeur(lt2, ext2);
                if (1 + Math.Max(v1, v2) < meilleure) meilleure = 1 + Math.Max(v1, v2);
            }
        }
        memo[cle] = meilleure;
        return meilleure;
    }

    var montre = Stopwatch.StartNew();
    int valeur = Valeur(new int[m], Enumerable.Range(0, ordres.Count).ToList());
    montre.Stop();
    return (valeur, memo.Count, msf.Distinct().Count(), montre.Elapsed.TotalSeconds);
}

// Recoupement : le meme minimax, sans aucune inference. L'etat est l'ENSEMBLE des permutations
// encore compatibles -- aucune cloture, donc aucune contrainte inventee n'est possible. Une
// reponse coupe cet ensemble en deux. Plus lent, mais il ne partage avec le calcul precedent
// que la definition de T* : si les deux tombent d'accord, l'inference ne fabrique rien.
int TEtoileExact((int U, int V)[] aretes, int n)
{
    int m = aretes.Length;
    var perms = ToutesPermutations(m);
    var positions = perms.Select(p =>
    {
        var d = new Dictionary<int, int>();
        for (int k = 0; k < p.Length; k += 1) d[p[k]] = k;
        return d;
    }).ToList();
    var foret = perms.Select(p => MsfDUnOrdre(aretes, p, n)).ToList();
    var memo = new Dictionary<string, int>();

    int Valeur(List<int> etat)
    {
        string cle = string.Join(",", etat);
        if (memo.TryGetValue(cle, out int cache)) return cache;
        if (etat.Select(i => foret[i]).Distinct().Count() == 1)
        {
            memo[cle] = 0;
            return 0;
        }
        int meilleure = 1000000000;
        for (int a = 0; a < m; a += 1)
        {
            for (int b = a + 1; b < m; b += 1)
            {
                var oui = etat.Where(i => positions[i][a] < positions[i][b]).ToList();
                if (oui.Count == 0 || oui.Count == etat.Count) continue;
                var non = etat.Where(i => positions[i][a] > positions[i][b]).ToList();
                int v1 = Valeur(oui);
                if (1 + v1 >= meilleure) continue;
                int v2 = Valeur(non);
                if (1 + Math.Max(v1, v2) < meilleure) meilleure = 1 + Math.Max(v1, v2);
            }
        }
        memo[cle] = meilleure;
        return meilleure;
    }

    return Valeur(Enumerable.Range(0, perms.Count).ToList());
}

var topologies = new (string Nom, int N, (int U, int V)[] Aretes)[]
{
    ("C3", 3, new[] { (0, 1), (1, 2), (0, 2) }),
    ("C4", 4, new[] { (0, 1), (1, 2), (2, 3), (3, 0) }),
    ("C5", 5, new[] { (0, 1), (1, 2), (2, 3), (3, 4), (4, 0) }),
    ("C6", 6, new[] { (0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 0) }),
    ("K4", 4, new[] { (0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3) }),
    ("K23", 5, new[] { (0, 3), (0, 4), (1, 3), (1, 4), (2, 3), (2, 4) }),
    ("papillon", 5, new[] { (0, 1), (1, 2), (0, 2), (2, 3), (3, 4), (2, 4) }),
    ("2 triangles", 6, new[] { (0, 1), (1, 2), (0, 2), (3, 4), (4, 5), (3, 5) }),
    ("etoile4", 5, new[] { (0, 1), (0, 2), (0, 3), (0, 4) }),
};

Console.WriteLine("T*(H) exact, par minimax sur les arbres de comparaison :");
Console.WriteLine("   {0,11} {1,2} {2,2} {3,7} {4,5} {5,3} {6,3} {7,10} {8,6}",
    "topologie", "n", "m", "m!", "#MSF", "lb", "T*", "tri fusion", "s");
var resultatsT = new List<(string Nom, int Lb, int T, int Ferme, int Etats)>();
foreach (var (nom, n, aretes) in topologies)
{
    var (valeurT, etats, nMsf, dt) = TEtoile(aretes, n);
    int lb = (int)Math.Ceiling(Math.Log2(Math.Max(nMsf, 1)));
    int ferme = aretes.Length > 1 ? FormeClose(aretes.Length) : 0;
    resultatsT.Add((nom, lb, valeurT, ferme, etats));
    Console.WriteLine($"   {nom,11} {n,2} {aretes.Length,2} {Factorielle(aretes.Length),7} {nMsf,5} {lb,3} {valeurT,3} {ferme,10} {dt,6:F2}");
}
Console.WriteLine();
Console.WriteLine("Etats memoises (ordre partiel clos + ensemble d'extensions) par topologie :");
foreach (var (nom, _, _, _, etats) in resultatsT)
{
    Console.WriteLine($"   {nom,11} : {etats,6} etats");
}
Console.WriteLine();
Console.WriteLine("Recoupement par le minimax exact, sans aucune inference :");
bool accord = true;
foreach (var (nom, n, aretes) in topologies)
{
    int exact = TEtoileExact(aretes, n);
    int mesure = resultatsT.First(r => r.Nom == nom).T;
    accord &= exact == mesure;
    Console.WriteLine($"   {nom,11} : avec inference {mesure,2}, sans inference {exact,2}");
}
Console.WriteLine(accord
    ? "   les deux calculs s'accordent sur les neuf topologies"
    : "   DESACCORD entre les deux calculs");

T*(H) exact, par minimax sur les arbres de comparaison :


     topologie  n  m      m!  #MSF  lb  T* tri fusion      s


            C3  3  3       6     3   2   2          3   0.00


            C4  4  4      24     4   2   3          5   0.00


            C5  5  5     120     5   3   4          8   0.02


            C6  6  6     720     6   3   5         11   0.82


... (22 outputs left out)

      papillon : avec inference  4, sans inference  4


   2 triangles : avec inference  4, sans inference  4


       etoile4 : avec inference  0, sans inference  0


   les deux calculs s'accordent sur les neuf topologies


### Lecture du résultat

Trois colonnes de la table se lisent ensemble : `#MSF` (le nombre d'arbres couvrants minimaux distincts que les permutations produisent), `lb` (la borne d'information `ceil(log2 #MSF)`, qui dit combien de bits il faut au minimum pour désigner l'un d'eux), et `T*` (la profondeur réellement atteinte par un arbre de comparaison optimal).

- **L'étoile est le cas dégénéré.** `etoile4` a 4 arêtes et un seul arbre couvrant possible : `#MSF = 1`, donc `lb = 0` et `T* = 0`. Aucune comparaison n'est nécessaire, et le tri fusion du même nombre d'arêtes en coûte déjà 5. Ce zéro est exact, pas un artefact du calcul.
- **Sur un cycle, `T*` suit la borne de près.** `C5` a 120 ordres possibles, 5 arbres couvrants minimaux (un par arête écartée) et `T* = 4`, contre `lb = 3` et 8 comparaisons pour trier. Choisir parmi 5 possibilités demande 3 bits ; 4 comparaisons suffisent à les distinguer.
- **Le graphe dense est le plus cher des neuf.** `K4` a 6 arêtes, 720 ordres, 16 arbres couvrants minimaux : `lb = 4` et `T* = 6`, pour 11 comparaisons de tri. L'écart avec la borne d'information y est le plus large : désigner l'un des 16 arbres demande 4 bits, et l'arbre optimal en dépense deux de plus.
- **Sur les neuf topologies, `lb <= T* <= tri fusion`.** L'écart entre `T*` et la borne d'information vaut 0 pour quatre topologies (`C3`, l'étoile, le papillon et les deux triangles), 1 pour trois (`C4`, `C5`, `K23`), et 2 pour deux (`C6`, `K4`) : la borne d'information n'est donc pas toujours atteignable.

Ce que cette table **ne** montre pas, et qui est le sujet du preprint : `T*` ici est calculé pour une topologie **fixée**. `T*(m, n)` est le **maximum** de cette quantité sur toutes les topologies à `m` arêtes et `n` sommets. Le maximum sur neuf graphes de six arêtes au plus ne dit rien de ce qui se passe à un million d'arêtes — mais il donne un étalon exact, et c'est cet étalon que la théorie doit retrouver.

Un détail de méthode : tous les poids sont supposés **distincts**. C'est la convention du preprint (« for every possible assignment of distinct weights »), et c'est ce qui rend l'arbre couvrant minimal unique.

Le second tableau mesure le coût de cette exactitude. Le nombre d'états mémoïsés reste petit ici — 15 pour `C3`, 119 pour `C4`, 1402 pour `C5`, puis 23929 pour `C6`, 58612 pour `K23`, 80748 pour `K4`, 16633 pour le papillon comme pour les deux triangles, et 1 pour l'étoile — parce que les états sont indexés par l'**ordre partiel** et non par l'historique des comparaisons : deux chemins qui imposent les mêmes relations retombent sur la même entrée. C'est ce quotient qui rend l'énumération des `720` permutations supportable, et c'est aussi ce qui la borne : la table grossit très vite avec `m`, ce qui interdit de dépasser six arêtes par cette voie.

Le troisième tableau est le contrôle, et il porte la valeur de tout ce qui précède. Pour chaque topologie, la colonne « avec inférence » est celle de la clôture, la colonne « sans inférence » celle du minimax exact sur les ensembles de permutations : **les deux s'accordent sur les neuf cas**. Une erreur de clôture ne se verrait pas dans une table unique — elle se lirait comme un résultat plus flatteur. Elle se voit ici, parce que le second calcul ne peut pas la commettre.

In [8]:
// Figure 3 du carnet Python, en texte : une barre par comparaison.
// Trois grandeurs par topologie -- borne d'information ceil(log2 #MSF), T* mesure, tri fusion.
string etiquetteBorne = "borne";
string etiquetteOptimal = "T*";
string etiquetteTri = "tri";

Console.WriteLine("Complexite de decision : ce que l'arbre optimal economise");
Console.WriteLine("(une barre = une comparaison)");
Console.WriteLine();
foreach (var (nom, lb, t, ferme, _) in resultatsT)
{
    Console.WriteLine($"   {nom,11}");
    Console.WriteLine($"       {etiquetteBorne,5} {lb,2} {new string('#', lb)}");
    Console.WriteLine($"       {etiquetteOptimal,5} {t,2} {new string('#', t)}");
    Console.WriteLine($"       {etiquetteTri,5} {ferme,2} {new string('#', ferme)}");
}
Console.WriteLine();
Console.WriteLine($"Sur les neuf topologies : {resultatsT.Count} cas, "
    + $"{resultatsT.Sum(r => r.Ferme)} comparaisons de tri, "
    + $"{resultatsT.Sum(r => r.T)} comparaisons pour l'arbre optimal, "
    + $"soit {resultatsT.Sum(r => r.Ferme - r.T)} epargnees.");

Complexite de decision : ce que l'arbre optimal economise


(une barre = une comparaison)


            C3


       borne  2 ##


          T*  2 ##


         tri  3 ###


... (29 outputs left out)

       borne  0 


          T*  0 


         tri  5 #####


Sur les neuf topologies : 9 cas, 76 comparaisons de tri, 33 comparaisons pour l'arbre optimal, soit 43 epargnees.


### Lecture du tableau

- **L'arbre optimal ne dépasse jamais le tri**, topologie par topologie : la barre du milieu n'est jamais plus haute que la barre de droite. Sur les six arêtes de `C6`, `K4`, `K23`, du papillon et des deux triangles, il utilise de `4` à `6` comparaisons là où le tri en demande `11` : entre un tiers et un peu plus de la moitié du coût. Sur les neuf topologies, le total est de `33` comparaisons pour l'arbre optimal contre `76` pour le tri du même nombre d'arêtes.
- **L'écart entre les deux barres de gauche est mince mais réel.** Il atteint deux comparaisons sur `C6` (borne 3, `T*` 5) et sur `K4` (borne 4, `T*` 6) : savoir *quel* arbre sort gagnant demande moins d'information que ce qu'un arbre de comparaison doit dépenser pour le savoir.
- **L'étoile est le seul cas où tout s'effondre.** Les deux premières barres sont nulles, la troisième vaut 5 : trier les arêtes d'un graphe qui n'a qu'un seul arbre couvrant est du travail entièrement inutile.

Le tableau donne donc un **fait** : sur ces graphes-là, le nombre de comparaisons nécessaire est une fraction du coût d'un tri. La question du preprint est de savoir si cette fraction reste bornée quand le nombre d'arêtes grandit — ou, ce qui revient au même, si `T*(m, n)` grandit comme `m`. Avant de lire la réponse, il faut comprendre la fonction qui figurait dans la meilleure borne connue jusqu'ici.

## 5. alpha(m, n) : l'inverse d'Ackermann, calculée exactement

La meilleure borne déterministe connue avant le preprint était, dans les termes des auteurs, `O(mα(m, n))`, « due to Chazelle [Cha00a] ». Et le texte précise ce qu'est ce `α` : « Here α is a two-parameter inverse-Ackermann function as in Tarjan's analysis of union–find [Tar75] ».

Une inverse d'Ackermann est souvent présentée comme « une fonction qui vaut au plus 4 en pratique ». C'est une affirmation. La cellule suivante la **calcule**.

### La convention employée, déclarée

L'inverse d'Ackermann n'a pas de définition unique : plusieurs conventions coexistent dans la littérature, et le preprint renvoie à l'analyse de Tarjan sans reprendre la formule. Pour qu'un calcul ait un sens, la convention doit être écrite. Ici, `A` est la fonction d'Ackermann-Peter (`A(0, n) = n + 1`, `A(m, 0) = A(m-1, 1)`, `A(m, n) = A(m-1, A(m, n-1))`), et deux conventions sont calculées côte à côte pour montrer que la conclusion ne dépend pas du choix :

- `alpha_>=` : le plus petit `i >= 1` tel que `A(i, m // n) >= ceil(log2 n)` ;
- `alpha_>` : le plus petit `i >= 1` tel que `A(i, m // n) > log2 n` (logarithme réel).

La cellule calcule aussi des valeurs exactes de `A` par la définition récursive, et compare les deux conventions aux points où elles risquent de différer.

Le calcul ne peut pas évaluer `A` sans borne : `A(4, 2)` a 19729 chiffres décimaux, et `A(5, ...)` n'est pas représentable. Comme `A` n'est utilisée que dans une comparaison à un logarithme, elle est donc **plafonnée** — dès qu'une valeur dépasse le plafond, elle est remplacée par `plafond + 1`, ce qui préserve toute comparaison et arrête la récursion. En C#, les valeurs exactes dont on a besoin (`A(4, 1) = 65533`, et le nombre de chiffres de `2^65533`) tiennent dans un `BigInteger`, ce qui permet de vérifier la borne au lieu de la citer.

In [9]:
// A(m, n) d'Ackermann-Peter, bornee a plafond + 1 : on ne la compare qu'a un logarithme.
// La memoisation porte sur (m, n) ; le plafond coupe la descente avant l'explosion.
int AckermannPlafonne(int m, int n, int plafond)
{
    var memo = new Dictionary<(int M, int N), int>();

    int A(int mm, int nn)
    {
        if (memo.TryGetValue((mm, nn), out int cache)) return cache;
        int valeur;
        if (mm == 0)
        {
            valeur = nn + 1;
        }
        else if (nn == 0)
        {
            valeur = A(mm - 1, 1);
        }
        else
        {
            int interne = A(mm, nn - 1);
            valeur = interne > plafond ? plafond + 1 : A(mm - 1, interne);
        }
        valeur = Math.Min(valeur, plafond + 1);
        memo[(mm, nn)] = valeur;
        return valeur;
    }

    return A(m, n);
}

int AckermannExact(int m, int n)
{
    if (m == 0) return n + 1;
    if (n == 0) return AckermannExact(m - 1, 1);
    return AckermannExact(m - 1, AckermannExact(m, n - 1));
}

// min{ i >= 1 : A(i, m / n) >= ceil(log2 n) } -- convention declaree, cf. texte.
int Alpha(long m, long n)
{
    int cible = (int)Math.Ceiling(Math.Log2(n));
    int i = 1;
    while (AckermannPlafonne(i, (int)Math.Max(m / n, 0), cible) < cible) i += 1;
    return i;
}

// Seconde convention : min{ i >= 1 : A(i, m / n) > log2 n }, logarithme reel.
int AlphaStrict(long m, long n)
{
    double cible = Math.Log2(n);
    int plafond = (int)Math.Ceiling(cible);
    int i = 1;
    while (AckermannPlafonne(i, (int)Math.Max(m / n, 0), plafond) <= cible) i += 1;
    return i;
}

Console.WriteLine("Valeurs exactes d'Ackermann (convention Ackermann-Peter) :");
foreach (var (mm, nn) in new[] { (1, 1), (2, 1), (2, 2), (3, 1), (3, 2), (3, 3), (3, 4) })
{
    Console.WriteLine($"   A({mm},{nn}) = {AckermannExact(mm, nn)}");
}
Console.WriteLine($"   A(4,1) = A(3, A(3,1)) = A(3,13) = 2^16 - 3 = {BigInteger.Pow(2, 16) - 3}");
Console.WriteLine();
Console.WriteLine("   {0,18} {1,9} {2,8}", "n (avec m = n)", "alpha >=", "alpha >");
foreach (long n in new long[] { 10, 100, 1000, 10000, 1000000, 1000000000, 1000000000000000000L })
{
    Console.WriteLine($"   {n,18} {Alpha(n, n),9} {AlphaStrict(n, n),8}");
}
Console.WriteLine();

string FormatSeuils(long[] seuils, Func<long, long, int> alpha)
{
    return string.Join(", ", seuils.Select(k => $"({k}, {alpha(k, k)})"));
}

Console.WriteLine("   seuils de la convention >= : "
    + FormatSeuils(new long[] { 8, 9, 32, 33, 8192, 8193 }, Alpha));
Console.WriteLine("   seuils de la convention >  : "
    + FormatSeuils(new long[] { 8, 9, 32, 33, 65536, 65537 }, AlphaStrict));

int chiffres = BigInteger.Pow(2, 65533).ToString().Length;
Console.WriteLine();
Console.WriteLine($"   2^65533 a {chiffres} chiffres.");
Console.WriteLine("   pour tout n <= 2^65533 et tout m >= n : A(4, m / n) >= A(4, 1) = 65533 >= log2 n,");
Console.WriteLine("   donc alpha <= 4 (convention >= ; stricte : pour tout n < 2^65533), et m*alpha(m,n) <= 4m.");
Console.WriteLine("   m < n change la valeur : n = 10^18, m = n / 2 : alpha >= "
    + $"{Alpha(1000000000000000000L / 2, 1000000000000000000L)}");

Valeurs exactes d'Ackermann (convention Ackermann-Peter) :


   A(1,1) = 3


   A(2,1) = 5


   A(2,2) = 7


   A(3,1) = 13


   A(3,2) = 29


... (13 outputs left out)

   2^65533 a 19728 chiffres.


   pour tout n <= 2^65533 et tout m >= n : A(4, m / n) >= A(4, 1) = 65533 >= log2 n,


   donc alpha <= 4 (convention >= ; stricte : pour tout n < 2^65533), et m*alpha(m,n) <= 4m.


   m < n change la valeur : n = 10^18, m = n / 2 : alpha >= 5


### Lecture du résultat

**D'abord les valeurs de `A`, calculées par la définition.** `A(1,1) = 3`, `A(2,1) = 5`, `A(3,1) = 13`, `A(3,3) = 61`, et surtout `A(4,1) = A(3, A(3,1)) = A(3,13) = 65533`. Le saut est net dès le quatrième niveau : passer de `A(3, ·)` à `A(4, 1)` fait déjà 65533. C'est ce saut qui rend l'inverse si petite.

**Ensuite `alpha`, colonne par colonne.**

- `alpha(10, 10) = 2` ; `alpha(100, 100) = 3` ; à partir de `n = 10 000` et jusqu'à `10^18`, `alpha = 4`.
- Les **seuils** disent la même chose autrement. Avec la convention `>=`, `alpha` passe à 2 à `n = 9`, reste 2 jusqu'à 32, passe à 3 à 33, reste 3 jusqu'à 8192, et passe à 4 à 8193. La convention stricte décale le dernier seuil : elle exige `n = 65536` pour passer à 4. Les deux conventions **coïncident** sur `n = 9`, `n = 33`, puis de `n = 1000` à `n = 10^18` ; elles diffèrent sur `n = 8` et `n = 32`, et par construction autour de leur dernier seuil (8193 contre 65536).
- **La borne « 4 » tient jusqu'à des tailles absurdes.** `2^65533` a 19728 chiffres. Pour tout `n <= 2^65533` et tout `m >= n`, `A(4, m / n) >= A(4,1) = 65533 >= log2 n`, donc `alpha <= 4` — et par conséquent `m·alpha(m, n) <= 4m`. Autrement dit : sur **toute** entrée dont le nombre de sommets tient en 19728 chiffres, la borne `O(m alpha)` vaut au plus `4m`. Ce n'est pas une approximation, c'est une conséquence de deux quantités mesurées ci-dessus (`A(4,1) = 65533` et la taille de `2^65533`).
- **`alpha` dépend des deux paramètres, pas seulement de `n`.** Avec `n = 10^18` et `m = n / 2`, le rapport `m / n` vaut 0, et la fonction doit monter jusqu'à `i = 5` : `alpha >= 5`. C'est la raison d'être du second paramètre : c'est `m/n`, la densité, qui décide de l'argument d'Ackermann.

### Ce que cela change — et ce que cela ne change pas

Voici le point le plus délicat du carnet, et il faut le dire sans le gonfler.

`alpha` est une fonction si lente que, sur toute entrée de taille réalisable, elle vaut au plus 4. La différence entre une borne `O(m alpha(m, n))` et une borne `O(m)` est donc, pour toute entrée qui existe, un **facteur constant** — au plus 4 sur le nombre de comparaisons. L'apport du preprint n'est pas « un algorithme plus rapide de 4 fois » : c'est un changement de **classe asymptotique**. Une borne `O(m alpha)` n'est pas linéaire au sens de la théorie, même si, à toute taille réalisable, elle en diffère d'un facteur 4 ; et c'est précisément ce « pas linéaire » qui laissait le problème ouvert depuis Chazelle.

Une réserve, mesurée : la constante **cachée** dans le `O(m)` du preprint n'est pas donnée explicitement dans le texte lu. Le preprint énonce `O(m)` et une constante `C_epsilon` dans son lemme technique (`epsilon` pouvant être pris à `1/8`), mais aucun nombre ne permet de dire « l'algorithme fait au plus `c·m` comparaisons avec `c = ...` ». Comparer les 4 comparaisons par arête de la borne `4m` à la constante réelle du nouvel algorithme est donc **hors d'atteinte** avec les moyens de ce carnet. C'est consigné comme ouvert en section 11.

Une dernière remarque, qui prépare la section suivante : cette borne `4m` n'est pas la borne d'un algorithme, c'est la borne d'une **structure** — l'union-find de Kruskal, dont l'analyse donne `alpha`. La cellule suivante mesure cette structure séparément.

In [10]:
Console.WriteLine("Union-Find de Kruskal : appels a trouver et pas de compression (fractionnement)");
Console.WriteLine("   {0,6} {1,7} {2,9} {3,8} {4,10} {5,10}",
    "n", "m", "appels", "pas", "pas/appel", "pas/arete");
foreach (int n in new[] { 500, 1000, 2000, 4000, 8000, 16000 })
{
    var (aretesT, wT) = GrapheAleatoire(n, 3.0, n);
    int mT = aretesT.Length;
    var ufT = new UnionFind(n);
    Kruskal(Enumerable.Range(0, mT).ToArray(), aretesT, wT, new Comparaisons(), ufT);
    Console.WriteLine($"   {n,6} {mT,7} {ufT.Appels,9} {ufT.Pas,8} {(double)ufT.Pas / ufT.Appels,10:F3} {(double)ufT.Pas / mT,10:F3}");
}

Union-Find de Kruskal : appels a trouver et pas de compression (fractionnement)


        n       m    appels      pas  pas/appel  pas/arete


      500    1500      3000     3966      1.322      2.644


     1000    3000      6000     8326      1.388      2.775


     2000    6000     12000    17948      1.496      2.991


     4000   12000     24000    37403      1.558      3.117


     8000   24000     48000    79025      1.646      3.293


    16000   48000     96000   168737      1.758      3.515


### Lecture du résultat — l'alpha de la structure n'est pas celui du problème

- **Deux appels à `Trouver` par arête, exactement.** Pour `n = 500`, `1500` arêtes donnent `3000` appels ; pour `n = 16000`, `48000` arêtes donnent `96000` appels. La raison est structurelle : chaque arête de l'ordre trié passe par `Unir`, qui cherche la racine de chacune de ses deux extrémités. Ce nombre ne dépend pas des poids.
- **Les pas de compression, eux, croissent.** De `1,322` pas par appel à `n = 500` à `1,758` à `n = 16000`, soit entre cinq et huit pour cent de plus à chaque doublement. Rapporté aux arêtes : de `2,64` à `3,52` pas par arête.

Deux mises en garde, toutes deux indispensables pour ne pas surinterpréter la table.

**Première mise en garde : la structure mesurée n'est pas celle du théorème `O(alpha)`.** L'union-find de ce carnet compresse les chemins (par fractionnement) mais **n'unit pas par rang ni par taille** : `Unir` accroche toujours la racine de `a` sous celle de `b`. La borne classique pour cette structure est `O(log n)` par opération, pas `O(alpha)`. La borne `alpha` de Tarjan vaut pour la compression **accompagnée** d'une union par taille ou par rang, et c'est une borne **amortie sur une séquence** d'opérations, pas un compteur que l'on peut lire appel par appel. La table mesure donc le coût d'une structure simplifiée : elle montre qu'il est petit et croît lentement, elle ne mesure pas `alpha`.

**Deuxième mise en garde : `alpha` est la borne d'une structure, pas celle du problème.** C'est exactement le piège que le preprint signale. Ses auteurs racontent que l'idée du temps linéaire a longtemps fait doute, « largely due to Ω(mα) lower bounds for related union-find data structures (see [Tar79b]), and variants of the MST problem [Pet06] ». Et le texte précise la variante en question : « Pettie [Pet06] ruled out one natural approach by proving that online verification, in which path-maximum queries arrive one at a time, indeed requires Ω(mα(m, n) + n) comparisons ». Autrement dit : un algorithme qui traiterait les requêtes une à une, en interrogeant une structure de chemins, doit payer `alpha`. Le théorème du preprint dit que le problème, lui, ne l'exige pas — à condition de ne pas travailler en ligne. La section 8 mesure ce que coûte, sur un jouet, la structure de chemins la plus naïve qui soit : une remontée pas à pas.

## 6. Le filtre : arêtes lourdes, arêtes légères, et les deux pièges

Tout le preprint tourne autour d'un geste : **jeter des arêtes sans perdre l'arbre**. C'est un théorème classique, que les auteurs rappellent ainsi :

> « For any forest F ⊆ G, let us call an edge F-heavy if it is heavier than every edge on the F-path between its endpoints, and F-light otherwise. By the cycle property, F-heavy edges can be discarded, and an adaptation of the verification algorithm identifies all of them in linear time [DRT92, KKT95]. »

Le preprint emploie aussi une seconde notion, définie à la section 1.2 juste avant son lemme principal :

> « for a graph G = (V, E) and a forest F ⊆ E, we say that an edge e ∈ E is supported by F, if the endpoints of e are connected by F≤w(e). In words, this is simply asking that if one adds the edge e to the forest F, then the resulting cycle uses edges of weight only ≤ w(e). »

Les deux notions se lisent sur une seule quantité : le maximum du chemin de `F` entre les extrémités de l'arête. Une arête hors de `F` est **lourde** si son poids dépasse **strictement** ce maximum, **légère** sinon. Elle est **supportée** si son poids est supérieur ou **égal** à ce maximum. À poids distincts, une arête hors de `F` est donc supportée exactement quand elle est lourde, et le lemme 1.2 compte sous `U(F)` les arêtes **non** supportées : ce sont celles que le filtre garde. Une arête de `F` elle-même est dans `F`. Une arête dont les extrémités ne sont pas reliées par `F` n'a pas de chemin, donc pas de verdict.

### Les deux pièges, et pourquoi ils comptent

1. **L'inégalité stricte est ce qui protège `F`.** Une arête n'est lourde que si `max(chemin) < w(e)` : une arête dont le poids **égale** le maximum du chemin n'est pas lourde. Ce cas n'est pas théorique — une arête de `F` est seule sur le chemin de `F` entre ses extrémités, donc son poids égale toujours ce maximum. Une règle qui ne garderait une arête que si `w(e) < max(chemin)` jetterait toutes les arêtes de `F`. La cellule ci-dessous montre la règle juste et cette règle fausse côte à côte, sur un triangle, le plus petit graphe qui contienne un cycle.
2. **Le chemin se lit dans le graphe courant, pas dans le graphe d'origine.** Quand l'algorithme a déjà contracté des composantes (phases de Borůvka), les extrémités d'une arête sont des **composantes** ; chercher le chemin entre les sommets d'origine donnerait un chemin qui traverse des arêtes déjà contractées, et un verdict faux. La classe `CheminMax` reçoit pour cela l'union-find courant : sans lui, c'est l'identité, et le chemin est cherché entre les sommets d'origine.

### Ce que cette section construit, et ce qu'elle ne construit pas

Ce qui est implémenté ici est un **classement exact**, arête par arête : pour chaque arête, un verdict — lourde, légère, dans `F`, ou sans chemin. Ce n'est pas le certificat du preprint. Le lemme 1.2 (énoncé à la section 9) garantit qu'une procédure déterministe construit, en `O_ε(m)` comparaisons de poids hors ses appels à des sous-instances, une forêt `F` pour laquelle `|F| + |U(F)| + Σ|Ei| <= εm + C_ε √(nm)`. Les auteurs résument son rôle en une phrase : elle « supports many of the original edges ». Classer toutes les arêtes d'un graphe et produire un certificat qui **réduit** le problème sont deux gestes différents.

Et le classement, tel qu'il est écrit ici, ne coûte pas une comparaison par arête : le maximum d'un chemin s'obtient en remontant ce chemin pas à pas. La cellule compte ces comparaisons à part, dans `Parcours` ; la section 8 mesure ce qu'elles coûtent quand `n` grandit.

In [11]:
// Pour une foret F : poids maximum sur le chemin de F entre deux sommets.
// Les extremites sont lues dans le graphe COURANT : si uf contracte deja des sommets
// (phases de Boruvka), le chemin est cherche entre les composantes, pas entre les
// sommets d'origine. Sans uf, c'est l'identite.
class CheminMax
{
    public Dictionary<int, int> Parent = new Dictionary<int, int>();
    public Dictionary<int, double> VersParent = new Dictionary<int, double>();
    public Dictionary<int, int> Profondeur = new Dictionary<int, int>();
    public Dictionary<int, int> Racine = new Dictionary<int, int>();
    public long Verdicts;
    public long Pas;
    public long Parcours;

    public CheminMax(List<int> F, (int U, int V)[] aretes, double[] w, UnionFind uf = null)
    {
        bool identite = uf == null;
        var adj = new Dictionary<int, List<(int Y, double W)>>();
        foreach (int e in F)
        {
            int u = aretes[e].U, v = aretes[e].V;
            if (!identite)
            {
                u = uf.Trouver(u);
                v = uf.Trouver(v);
            }
            if (!adj.ContainsKey(u)) adj[u] = new List<(int Y, double W)>();
            if (!adj.ContainsKey(v)) adj[v] = new List<(int Y, double W)>();
            adj[u].Add((v, w[e]));
            adj[v].Add((u, w[e]));
        }
        foreach (int s in adj.Keys.ToList())
        {
            if (Parent.ContainsKey(s)) continue;
            Parent[s] = s;
            Profondeur[s] = 0;
            Racine[s] = s;
            var pile = new Stack<int>();
            pile.Push(s);
            while (pile.Count > 0)
            {
                int x = pile.Pop();
                foreach (var (y, wy) in adj[x])
                {
                    if (Parent.ContainsKey(y)) continue;
                    Parent[y] = x;
                    VersParent[y] = wy;
                    Profondeur[y] = Profondeur[x] + 1;
                    Racine[y] = s;
                    pile.Push(y);
                }
            }
        }
    }

    public bool Relies(int u, int v)
        => Racine.ContainsKey(u) && Racine.ContainsKey(v) && Racine[u] == Racine[v];

    // Poids le plus lourd sur le chemin de F entre u et v (appele seulement si reliees).
    // Un chemin de k aretes coute k - 1 comparaisons de poids : la premiere arete devient
    // le maximum sans comparaison, chacune des suivantes est comparee au maximum courant.
    // Ces comparaisons ont lieu ENTRE ARETES DE F : l'instrument cmp ne les voit pas,
    // elles sont comptees ici, dans Parcours. Pas compte les aretes parcourues, Verdicts
    // les appels.
    public double Maximum(int u, int v)
    {
        if (u == v) return 0.0;
        Verdicts += 1;
        if (Profondeur[u] < Profondeur[v]) (u, v) = (v, u);
        double maxi = VersParent[u];
        Pas += 1;
        u = Parent[u];
        while (u != v)
        {
            if (Profondeur[u] < Profondeur[v]) (u, v) = (v, u);
            Pas += 1;
            Parcours += 1;
            if (VersParent[u] > maxi) maxi = VersParent[u];
            u = Parent[u];
        }
        return maxi;
    }
}

// "F" | "lourde" | "legere" | "sans_chemin". Lourde ssi plus lourde que TOUTE arete du chemin.
// Lourde = F-heavy du preprint (jetable) ; legere = F-light (gardee) ; ni l'un ni l'autre si
// les extremites ne sont pas reliees par F.
Dictionary<int, string> Classifier(int[] E, (int U, int V)[] aretes, double[] w, List<int> F,
    CheminMax chemin, Comparaisons cmp, UnionFind uf = null)
{
    var dansF = new HashSet<int>(F);
    var resultat = new Dictionary<int, string>();
    foreach (int e in E)
    {
        int u = aretes[e].U, v = aretes[e].V;
        if (uf != null)
        {
            u = uf.Trouver(u);
            v = uf.Trouver(v);
        }
        if (dansF.Contains(e)) resultat[e] = "F";
        else if (!chemin.Relies(u, v)) resultat[e] = "sans_chemin";
        else if (cmp.Inferieur(chemin.Maximum(u, v), w[e])) resultat[e] = "lourde";
        else resultat[e] = "legere";
    }
    return resultat;
}

// La regle fausse : garder si strictement plus legere que le maximum du chemin.
Dictionary<int, string> ClassifierPiege(int[] E, (int U, int V)[] aretes, double[] w,
    List<int> F, CheminMax chemin, Comparaisons cmp)
{
    var resultat = new Dictionary<int, string>();
    foreach (int e in E)
    {
        int u = aretes[e].U, v = aretes[e].V;
        if (!chemin.Relies(u, v)) resultat[e] = "sans_chemin";
        else if (cmp.Inferieur(w[e], chemin.Maximum(u, v))) resultat[e] = "legere";
        else resultat[e] = "lourde";
    }
    return resultat;
}

string FormatVerdicts(Dictionary<int, string> verdicts)
    => string.Join(" ", verdicts.OrderBy(kv => kv.Key).Select(kv => $"{kv.Key}={kv.Value}"));

var aretesPetit = new[] { (0, 1), (1, 2), (0, 2) };
var wPetit = new[] { 1.0, 2.0, 3.0 };
var FPetit = new List<int> { 0, 1 };
var cheminPetit = new CheminMax(FPetit, aretesPetit, wPetit);
var juste = Classifier(new[] { 0, 1, 2 }, aretesPetit, wPetit, FPetit, cheminPetit, new Comparaisons());
var faux = ClassifierPiege(new[] { 0, 1, 2 }, aretesPetit, wPetit, FPetit, cheminPetit, new Comparaisons());
int jetees = faux.Values.Count(c => c == "lourde");
Console.WriteLine("Triangle 0-1 (1), 1-2 (2), 0-2 (3) ; F = les deux aretes les plus legeres :");
Console.WriteLine($"   regle juste : {FormatVerdicts(juste)}");
Console.WriteLine($"   regle fausse : {FormatVerdicts(faux)} -> elle jette les {jetees} aretes, dont les 2 de F : il ne reste aucun arbre");

Triangle 0-1 (1), 1-2 (2), 0-2 (3) ; F = les deux aretes les plus legeres :


   regle juste : 0=F 1=F 2=lourde


   regle fausse : 0=lourde 1=lourde 2=lourde -> elle jette les 3 aretes, dont les 2 de F : il ne reste aucun arbre


### Lecture du résultat

Sur le triangle (arêtes de poids 1, 2 et 3, avec `F` = les deux plus légères) :

- **La règle juste** rend `0=F 1=F 2=lourde` : les deux arêtes de `F` sont protégées par leur appartenance à `F`, et la troisième est reconnue lourde parce que son chemin dans `F` a un maximum de 2, strictement inférieur à son poids 3. Le cycle `0-1-2` n'utilise que des arêtes plus légères : par la propriété du cycle, l'arête `0-2` ne peut être dans aucun arbre couvrant minimal.
- **La règle fausse** rend `0=lourde 1=lourde 2=lourde` : elle jette les trois arêtes, dont les deux de `F`. Il ne reste plus rien pour construire un arbre.

La règle fausse porte en réalité **deux** défauts, et c'est ce que le triangle rend visible d'un coup :

1. **Elle ne protège pas les arêtes de `F`.** Une arête déjà retenue dans la forêt ne se rejette jamais : c'est le seul objet que le filtre doit conserver par construction. La règle juste teste l'appartenance à `F` avant toute comparaison ; la règle fausse compare toutes les arêtes, celles de `F` comprises.
2. **Elle lit l'égalité comme de la lourdeur.** Sa comparaison demande `w(e) < max(chemin)` pour garder ; une arête dont le poids **égale** le maximum du chemin est donc rejetée. Or la définition du preprint est explicite — lourde signifie « heavier than **every** edge on the F-path », donc strictement plus lourde que toutes ; égaler le maximum ne suffit pas. Et l'égalité est la situation habituelle d'une arête de `F` : seule sur le chemin de `F` entre ses extrémités, elle en est le maximum. C'est ainsi que la règle fausse jette les deux arêtes de `F`. Sur la troisième arête, plus lourde que le maximum de son chemin, les deux règles sont d'accord : elle est lourde.

C'est pourquoi `Classifier` teste trois choses dans cet ordre : l'appartenance à `F`, l'existence d'un chemin, puis la comparaison stricte. Sans la comparaison stricte, une arête de `F` serait jugée lourde ; sans le test du chemin, on chercherait un maximum entre deux sommets que `F` ne relie pas.

**Combien coûte un verdict ?** Côté instrument, **une** comparaison : le maximum de son chemin contre son poids. Mais ce maximum, `Maximum` l'obtient en remontant le chemin de `F`. La première arête du chemin devient le maximum sans rien comparer ; chacune des suivantes est comparée au maximum courant. Un chemin de `k` arêtes de `F` coûte donc `k − 1` comparaisons — entre poids d'arêtes de `F`, donc invisibles pour l'instrument. Ce sont pourtant des comparaisons de poids, au sens du modèle de la section 1. La classe les compte donc à part, dans trois champs : `Verdicts` (requêtes qui parcourent un chemin), `Pas` (arêtes de `F` parcourues) et `Parcours` (comparaisons entre poids d'arêtes de `F`, soit `Pas − Verdicts`). Sur le triangle, la remontée est courte : deux pas, une comparaison. Elle ne le reste pas quand la forêt grandit : c'est ce que mesurent les deux sections suivantes.

Reste le piège de la petite taille : ce triangle ne prouve pas que la règle est correcte en général, il prouve seulement qu'elle n'est pas **trivialement** fausse. Le fait général est le théorème du cycle, cité plus haut ; ce qui est mesuré ici est que l'implémentation ne s'en écarte pas sur un cas où deux implémentations plausibles divergent.

## 7. La route randomisée : échantillonner puis filtrer

Avant le résultat déterministe, la route connue passait par le hasard. Le preprint la résume en deux phrases (section 1, puis section 1.2) :

> « Klein and Tarjan [KT94], and later Karger, Klein, and Tarjan [KKT95], gave a randomized algorithm that always returns an MST and runs in linear time both in expectation and with high probability. »

> « Their algorithm alternates between so-called "Borůvka phases", which shrink the number of vertices, with the new idea of sampling and filtering, which shrink the number of edges. The crucial lemma which underlies their result shows that if each edge is sampled independently with probability 0 < p ≤ 1 and F is the sample's minimum spanning forest, then the expected number of F-light edges of G is at most n/p. This idea of using Borůvka phases to reduce vertices, and "filtering" to reduce edges, will also re-appear in the algorithm we present below (though the filtering will naturally require an entirely different approach to achieve a deterministic guarantee). »

### Ce que dit le lemme, et ce qu'il ne dit pas

Le lemme est une borne sur une **espérance**. Il affirme que, en moyenne sur les tirages de l'échantillon, il reste au plus `n/p` arêtes non lourdes. Il n'affirme **pas** que chaque tirage respecte `n/p` : un tirage donné peut dépasser la borne, et c'est précisément ce que la mesure ci-dessous va montrer. C'est le genre d'écart qui sépare un énoncé probabiliste d'une garantie déterministe, et c'est la raison pour laquelle la route randomisée ne suffisait pas.

La cellule suivante mesure, sur deux densités et trois probabilités :

- le nombre moyen d'arêtes qui **survivent** au filtre (celles qui ne sont pas classées lourdes) ;
- son erreur type, sur 2000 tirages indépendants ;
- le rapport à la borne `n/p` ;
- le minimum, le maximum, et le nombre de tirages qui **dépassent** la borne.

In [12]:
Console.WriteLine("Lemme d'echantillonnage : E[aretes F-legeres] <= n/p, mesure sur n = 100.");
var tiragesHisto = new List<int>();
foreach (double ratio7 in new[] { 3.0, 10.0 })
{
    int n7 = 100;
    var (aretes7, w7) = GrapheAleatoire(n7, ratio7, 11);
    int m7 = aretes7.Length;
    Console.WriteLine($"   n={n7} m={m7} (m/n={ratio7:F0})");
    Console.WriteLine("   " + "p".PadLeft(5) + "n/p".PadLeft(7) + "moyenne".PadLeft(8)
        + "erreur type".PadLeft(12) + "ratio".PadLeft(6) + "min".PadLeft(4)
        + "max".PadLeft(4) + ">borne".PadLeft(7));
    foreach (double p in new[] { 0.5, 0.3, 0.2 })
    {
        var rng = new Random(7);
        int R = 2000;
        var valeurs = new List<int>();
        for (int tirage = 0; tirage < R; tirage += 1)
        {
            var echantillon = Enumerable.Range(0, m7).Where(e => rng.NextDouble() < p).ToArray();
            var F = Kruskal(echantillon, aretes7, w7, new Comparaisons(), new UnionFind(n7));
            var classes = Classifier(Enumerable.Range(0, m7).ToArray(), aretes7, w7, F,
                new CheminMax(F, aretes7, w7), new Comparaisons());
            valeurs.Add(classes.Values.Count(c => c != "lourde"));
        }
        double borne = n7 / p;
        double moyenne = valeurs.Average();
        double variance = valeurs.Sum(v => (v - moyenne) * (v - moyenne)) / (R - 1);
        int depassent = valeurs.Count(v => v > borne);
        Console.WriteLine($"   {p,5:F2} {borne,7:F0} {moyenne,8:F2} {Math.Sqrt(variance / R),12:F2}"
            + $" {moyenne / borne,6:F3} {valeurs.Min(),4} {valeurs.Max(),4} {depassent,7}");
        if (ratio7 == 10.0 && p == 0.5)
        {
            tiragesHisto = valeurs;
        }
    }
}
Console.WriteLine();
Console.WriteLine("   decomposition d'un tirage (n=100, m=1000, p=0.5) :");
var (aretesD, wD) = GrapheAleatoire(100, 10.0, 11);
var rngD = new Random(3);
var echD = Enumerable.Range(0, aretesD.Length).Where(e => rngD.NextDouble() < 0.5).ToArray();
var FD = Kruskal(echD, aretesD, wD, new Comparaisons(), new UnionFind(100));
var cmpD = new Comparaisons();
var cheminD = new CheminMax(FD, aretesD, wD);
var classesD = Classifier(Enumerable.Range(0, aretesD.Length).ToArray(), aretesD, wD, FD, cheminD, cmpD);
string resumeD = string.Join(", ", classesD.Values.GroupBy(c => c).OrderBy(g => g.Key)
    .Select(g => $"{g.Key}={g.Count()}"));
Console.WriteLine($"    {resumeD} ; aretes comparees au maximum de leur chemin : {cmpD.N}");
Console.WriteLine($"    comparaisons internes de ces parcours de chemin : {cheminD.Parcours}");
Console.WriteLine($"    longueur moyenne d'un chemin parcouru (en aretes de F) : {cheminD.Pas / (double)cheminD.Verdicts:F2}");

Lemme d'echantillonnage : E[aretes F-legeres] <= n/p, mesure sur n = 100.


   n=100 m=300 (m/n=3)


       p    n/p moyenne erreur type ratio min max >borne


    0.50     200   186.00         0.26  0.930  147  228     214


    0.30     333   261.04         0.31  0.783  213  295       0


    0.20     500   292.05         0.13  0.584  259  300       0


... (5 outputs left out)

   decomposition d'un tirage (n=100, m=1000, p=0.5) :


    F=99, legere=92, lourde=809 ; aretes comparees au maximum de leur chemin : 901


    comparaisons internes de ces parcours de chemin : 6836


    longueur moyenne d'un chemin parcouru (en aretes de F) : 8.59


### Lecture du résultat

**L'espérance est bien sous la borne, partout.** Les six lignes donnent des rapports `moyenne / (n/p)` de `0,584` à `0,991` : jamais au-dessus. L'erreur type est petite (`0,13` à `0,93`), ce qui rend les moyennes fiables.

**Chaque tirage, en revanche, ne respecte rien.** À densité 3 et `p = 1/2`, `214` tirages sur `2000` dépassent la borne de 200, avec un maximum à `228`. À densité 10 et `p = 1/2`, c'est `817` sur `2000` — plus de quatre tirages sur dix — avec un maximum à `266`. Le lemme est donc exactement ce qu'il annonce : une borne sur une **moyenne**, pas un garde-fou par tirage.

**La borne n'est pas atteinte par le haut non plus.** À densité 3 et `p = 1/5`, la moyenne vaut `292` pour une borne de `500` : la borne est lâche dans ce régime. Elle est en revanche presque serrée à densité 10, où les rapports montent à `0,973`, `0,985` et `0,991`.

**Ce que coûte la mesure elle-même.** Sur le tirage décomposé (`m = 1000`), `901` arêtes sont comparées au maximum de leur chemin — une comparaison chacune. Elles se répartissent en `809` lourdes et `92` légères ; les `99` arêtes de `F` ne sont pas comparées. Mais le maximum d'un chemin ne tombe pas du ciel : `Maximum` le trouve en remontant le chemin pas à pas. Sur ce tirage, ces remontées ont coûté `6836` comparaisons de poids entre arêtes de `F` — plus de sept fois le classement lui-même. La cause se lit sur la dernière ligne : les chemins parcourus ont en moyenne `8,59` arêtes de `F`, soit `7,59` comparaisons internes par arête classée. L'énoncé cité en section 6, « an adaptation of the verification algorithm identifies all of them in linear time », repose donc sur un **algorithme de vérification**, pas sur une remontée naïve. La section 8 mesure ce que coûte la remontée quand `n` grandit.

Ce constat — moyenne sûre, tirage non sûr — est ce qui justifie la section suivante : comparer trois façons de **choisir** l'échantillon, dont deux déterministes.

In [13]:
// Version texte de la figure : un noyau C# n'a pas de matplotlib, l'histogramme se lit en caracteres.
double moyenneHisto = tiragesHisto.Average();
int auDessus = tiragesHisto.Count(v => v > 200);
int minHisto = tiragesHisto.Min();
int maxHisto = tiragesHisto.Max();
int nbSeaux = 28;
double largeurSeau = (maxHisto - minHisto + 1) / (double)nbSeaux;
var compte = new int[nbSeaux];
foreach (int v in tiragesHisto)
{
    int seau = (int)((v - minHisto) / largeurSeau);
    if (seau >= nbSeaux) seau = nbSeaux - 1;
    compte[seau] += 1;
}
int maxCompte = compte.Max();
int largeurBarre = 44;
double tiragesParDiese = (double)maxCompte / largeurBarre;
Console.WriteLine("Ce que la borne controle : l'esperance, pas chaque tirage");
Console.WriteLine("2000 tirages, p = 1/2, n = 100, m = 1000 ; aretes F-legeres qui survivent au filtre");
Console.WriteLine($"un '#' vaut {tiragesParDiese:F2} tirages ; borne de l'esperance n/p = 200, moyenne mesuree = {moyenneHisto:F1}");
Console.WriteLine();
for (int seau = 0; seau < nbSeaux; seau += 1)
{
    int debut = minHisto + (int)Math.Round(seau * largeurSeau);
    int barre = (int)Math.Round(compte[seau] / tiragesParDiese);
    string note = "";
    if (debut <= 200 && debut + largeurSeau > 200)
    {
        note = "   <- borne n/p = 200";
    }
    else if (debut > 200)
    {
        note = "   | hors borne";
    }
    Console.WriteLine($"{debut,5} +{new string('#', barre)}{note}");
}
Console.WriteLine($"{maxHisto + 1,5} +  (fin de l'axe)");
Console.WriteLine();
Console.WriteLine($"{auDessus} tirages sur 2000 depassent la borne ; minimum {minHisto}, maximum {maxHisto}");

Ce que la borne controle : l'esperance, pas chaque tirage


2000 tirages, p = 1/2, n = 100, m = 1000 ; aretes F-legeres qui survivent au filtre


un '#' vaut 5.59 tirages ; borne de l'esperance n/p = 200, moyenne mesuree = 198.3


  151 +


  155 +


  159 +#


... (23 outputs left out)

  259 +   | hors borne


  263 +   | hors borne


  267 +  (fin de l'axe)


817 tirages sur 2000 depassent la borne ; minimum 151, maximum 266


### Lecture de la figure

L'histogramme porte les `2000` tirages de la ligne `p = 1/2` à densité 10.

- La distribution a la forme d'une **cloche**, centrée sur `198`, à gauche de la borne `200`. C'est l'illustration du lemme : en moyenne, le filtre laisse passer moins de `n/p` arêtes.
- La zone marquée `hors borne` à droite de la borne contient `817` tirages sur `2000`. La borne de l'espérance coupe donc la cloche : elle n'est pas une **barrière**, c'est une **moyenne majorée**.
- La largeur de la cloche va d'environ `151` à `266`. Aucun tirage ne s'approche des `1000` arêtes du graphe : le filtre a bien fait son travail sur chaque tirage, simplement pas avec la même marge.

Un algorithme randomisé peut se satisfaire de cette garantie : il sait qu'**en moyenne** le nombre d'arêtes diminue, et il borne son espérance de coût. Un algorithme déterministe ne le peut pas : il doit être sûr que le nombre d'arêtes diminue **à chaque** appel, sur **toute** entrée. La question devient alors : sans hasard, comment choisir un échantillon dont on puisse garantir quelque chose ?

### Trois façons de choisir l'échantillon — déterministe ou randomisé, montré par une expérience

La question « déterministe ou randomisé » se laisse réduire à un geste concret : **comment choisir les 500 arêtes de l'échantillon parmi les 1000** ? Trois réponses sont comparées sur le même graphe (`n = 100`, `m = 1000`).

| Choix | Comparaisons pour choisir | Regarde-t-il les poids ? | Garantie |
|---|---|---|---|
| **Aveugle** : les 500 premières arêtes par indice | 0 | non | aucune |
| **Aléatoire** : chaque arête avec probabilité `1/2` | 0 | non | en espérance seulement |
| **Tri complet** : les 500 plus légères | un tri | oui | déterministe |

Chaque choix est éprouvé sur **deux** jeux de poids :

- des **poids ordinaires** (permutation fixée d'avance de `1..1000`) ;
- des **poids adverses**, construits en connaissant le choix aveugle : les 500 arêtes de plus petit indice reçoivent les poids `501..1000`, les autres les poids `1..500`. L'échantillon aveugle est alors constitué des **500 arêtes les plus lourdes** du graphe.

La colonne mesurée est le nombre d'arêtes **non lourdes** qui survivent au filtre : moins il y en reste, plus le filtre a servi.

La règle du jeu est à lire précisément. L'aléatoire et l'aveugle ont le même coût de choix — zéro comparaison — et le même aveuglement devant les poids. Seul le hasard de l'un le protège : un adversaire peut concevoir des poids qui ruinent un choix **fixe**, il ne peut pas en concevoir qui ruinent **en moyenne** un choix aléatoire. Le tri complet, lui, est déterministe et protégé, mais il paie pour l'être.

In [14]:
Console.WriteLine("Trois facons de choisir l'echantillon (n = 100, m = 1000, |S| = 500) :");
int n35 = 100;
var (aretes35, w35) = GrapheAleatoire(n35, 10.0, 21);
int m35 = aretes35.Length;
int moitie35 = m35 / 2;
var wAdv = new double[m35];
for (int e = 0; e < m35; e += 1)
{
    wAdv[e] = e < moitie35 ? (double)(moitie35 + 1 + e) : (double)(1 + e - moitie35);
}

int LegeresDe(List<int> F, double[] poids)
{
    var cl = Classifier(Enumerable.Range(0, m35).ToArray(), aretes35, poids, F,
        new CheminMax(F, aretes35, poids), new Comparaisons());
    return cl.Values.Count(c => c != "lourde");
}

var tous35 = Enumerable.Range(0, m35).ToArray();
var premiers35 = Enumerable.Range(0, moitie35).ToArray();

// 1. Choix aveugle : les 500 premieres aretes par indice, sans regarder les poids.
var fAdv = Kruskal(premiers35, aretes35, wAdv, new Comparaisons(), new UnionFind(n35));
var fOrd = Kruskal(premiers35, aretes35, w35, new Comparaisons(), new UnionFind(n35));
int legersAveugleAdv = LegeresDe(fAdv, wAdv);
int legersAveugleOrd = LegeresDe(fOrd, w35);

// 2. Choix aleatoire : chaque arete avec probabilite 1/2, moyenne sur 2000 tirages.
var rngChoix = new Random(5);
int R35 = 2000;
double sommeAdv = 0.0;
double sommeOrd = 0.0;
for (int t = 0; t < R35; t += 1)
{
    var ech = tous35.Where(e => rngChoix.NextDouble() < 0.5).ToArray();
    sommeAdv += LegeresDe(Kruskal(ech, aretes35, wAdv, new Comparaisons(), new UnionFind(n35)), wAdv);
    sommeOrd += LegeresDe(Kruskal(ech, aretes35, w35, new Comparaisons(), new UnionFind(n35)), w35);
}

// 3. Choix informe : tri complet des poids, on garde les 500 plus legeres.
var cmpAdv = new Comparaisons();
var cmpOrd = new Comparaisons();
var triAdv = TriFusion(tous35, wAdv, cmpAdv);
var triOrd = TriFusion(tous35, w35, cmpOrd);
var fTriAdv = Kruskal(triAdv.Take(moitie35).ToArray(), aretes35, wAdv, new Comparaisons(), new UnionFind(n35));
var fTriOrd = Kruskal(triOrd.Take(moitie35).ToArray(), aretes35, w35, new Comparaisons(), new UnionFind(n35));
int legersTriAdv = LegeresDe(fTriAdv, wAdv);
int legersTriOrd = LegeresDe(fTriOrd, w35);

Console.WriteLine("   " + "choix".PadLeft(34) + "pour choisir".PadLeft(13)
    + "poids adverses".PadLeft(15) + "poids ordinaires".PadLeft(17));
void LigneChoix(string libelle, string cout, string adv, string ordinaire)
{
    Console.WriteLine($"   {libelle,34} {cout,13} {adv,15} {ordinaire,17}");
}
LigneChoix("aveugle (500 premiers indices)", "0",
    legersAveugleAdv.ToString(), legersAveugleOrd.ToString());
LigneChoix($"aleatoire p = 1/2 (moyenne {R35})", "0",
    $"{sommeAdv / R35:F2}", $"{sommeOrd / R35:F2}");
LigneChoix("tri complet (500 plus legere)", $"{cmpAdv.N} / {cmpOrd.N}",
    legersTriAdv.ToString(), legersTriOrd.ToString());

Trois facons de choisir l'echantillon (n = 100, m = 1000, |S| = 500) :


                                choix pour choisir poids adverses poids ordinaires


       aveugle (500 premiers indices)             0             599               205


     aleatoire p = 1/2 (moyenne 2000)             0          198.22            198.17


        tri complet (500 plus legere)   4932 / 8732              99                99


### Lecture du résultat — ce que l'expérience établit

La table se lit ligne par ligne, en comparant les deux colonnes de poids. Dans la colonne « pour choisir », les deux nombres de la dernière ligne suivent l'ordre des deux colonnes de poids : `4932` comparaisons pour trier les poids adverses, `8732` pour trier les poids ordinaires.

| Choix | Pour choisir | Poids adverses | Poids ordinaires |
|---|---|---|---|
| aveugle | 0 | **599** | 205 |
| aléatoire | 0 | **198,22** | 198,17 |
| tri complet | 4932 / 8732 | **99** | 99 |

1. **Un choix déterministe et aveugle est ruinable, sans qu'on lui coûte une comparaison.** Sur des poids ordinaires, l'échantillon aveugle laisse passer `205` arêtes — au niveau de la borne `n/p = 200`. Sur des poids construits **contre lui**, il en laisse passer `599` : près du triple. Le mécanisme est lisible : l'échantillon est fait des 500 arêtes les plus lourdes, leur forêt ne sert à rien pour écarter les arêtes légères, et les 500 arêtes légères traversent toutes le filtre, plus `99` autres.
2. **Le hasard, au même coût, résiste.** L'échantillon aléatoire coûte lui aussi `0` comparaison et ne regarde pas plus les poids ; sur les **mêmes** poids adverses, il laisse passer `198,22` arêtes en moyenne — la même valeur (`198,17`) que sur des poids ordinaires. Le lemme d'échantillonnage le garantit : sa borne porte sur l'espérance, quels que soient les poids. Un adversaire qui connaît l'algorithme ne peut donc pas construire de poids qui ruinent la **moyenne** d'un tirage aléatoire.
3. **La détermination informée se paie.** Trier et garder les 500 plus légères donne le meilleur filtre, `99` arêtes survivantes dans les deux cas — exactement la taille de la forêt échantillon. Mais il a fallu `8732` comparaisons sur des poids ordinaires, soit `97 %` du pire cas `C(1000) = 8977` d'un tri de 1000 éléments. Le filtre est gratuit ; c'est le **choix** qui coûte un tri.

**Conclusion de l'expérience.** À trois ingrédients — coût du choix, regard sur les poids, garantie — aucune des trois méthodes n'a tout : l'aveugle est gratuit et sans garantie, l'aléatoire est gratuit et garanti en espérance, le tri est garanti et coûteux. C'est précisément le vide que comble le preprint : une façon **déterministe** de produire la forêt qui sert à filtrer, **garantie sur toute entrée**, pour un nombre linéaire de comparaisons (lemme 1.2, section 9). Le théorème n'est pas reproduit ici ; l'expérience montre seulement pourquoi les deux voies évidentes ne suffisaient pas.

## 8. Le squelette : Borůvka, échantillon, filtre, socle

L'algorithme du preprint n'est pas reproduit ici : il repose sur une procédure de certificat déterministe (lemme 1.2) dont l'énoncé est donné en section 9, et qui n'est pas implémentée dans ce carnet. Ce que cette section **mesure**, c'est le **squelette** commun à la route randomisée des années 1990 et à l'algorithme du preprint, tel que les auteurs le décrivent : « Borůvka phases » pour réduire les sommets, « filtering » pour réduire les arêtes, et on recommence.

**Avertissement, à lire avant la table.** Le programme ci-dessous est un **jouet inspiré de Karger–Klein–Tarjan** : il échantillonne **au hasard**. Il est donc randomisé, et ce n'est **pas** l'algorithme du preprint. Il sert à voir le squelette fonctionner et à mesurer ce que chaque étape apporte ; il ne démontre rien de déterministe.

### Le squelette, en quatre gestes

1. **Cas de base (socle).** S'il reste peu d'arêtes (`m <= 1,5 n`), on termine par un algorithme simple.
2. **Phases de Borůvka.** Chaque composante garde son arête sortante la plus légère, qui appartient à l'arbre ; on contracte. Le texte des auteurs : « every phase at least halves the number of vertices ».
3. **Échantillon et récursion.** On tire chaque arête avec probabilité `1/2`, on calcule récursivement la forêt couvrante minimale de l'échantillon.
4. **Filtre et récursion.** On écarte les arêtes lourdes pour cette forêt (section 6), et on recommence sur les survivantes.

### Trois compteurs, parce que l'instrument ne voit pas tout

L'instrument `cmp` compte les comparaisons faites **à travers lui** : le tri du socle, les choix des phases de Borůvka, et — dans le filtre — une comparaison par arête classée (le maximum de son chemin contre son poids). Il ne voit pas les comparaisons internes de `CheminMax.Maximum`, qui remonte le chemin de `F` en comparant deux poids d'arêtes de `F` à chaque pas après le premier (section 6). Ce sont pourtant des comparaisons de poids, au sens du modèle de la section 1.

`CheminMax` tient donc trois champs, nuls pour chaque nouveau filtre et additionnés sur toute l'exécution : `Verdicts` (requêtes qui parcourent un chemin), `Pas` (arêtes de `F` parcourues) et `Parcours` (comparaisons entre poids d'arêtes de `F`). Un chemin de `k` arêtes coûte `k − 1` comparaisons, d'où l'identité `Parcours = Pas − Verdicts`, contrôlée par un `Exiger` sur la trace. Toutes les colonnes sont ramenées au nombre d'arêtes du graphe :

- **`jouet/K`** : comparaisons vues par l'instrument, socle Kruskal ;
- **`parc/K`** : comparaisons internes des remontées de chemin, pour la même exécution ;
- **`K+parc`** : leur somme — le coût complet, dans le modèle de comparaison ;
- **`jouet/R`** : comparaisons vues par l'instrument quand le socle range les arêtes par poids entier, **hors** modèle de comparaison. Ce n'est pas un algorithme du modèle : c'est une référence qui montre ce que le squelette coûterait si le cas de base était gratuit.

La ligne « origine » de la trace donne, pour une exécution, le nombre de verdicts par arête et la longueur moyenne des chemins parcourus. Un second tableau décompose `parc/K` en ces deux facteurs ; un troisième, à `n = 4000` fixé, fait varier la densité.

Un contrôle accompagne chaque mesure : l'arbre rendu doit avoir exactement les mêmes poids que celui de Kruskal (`Exiger`).

In [15]:
// Socle de substitution, HORS modele de comparaison : range les aretes par poids entier.
List<int> SocleTriParBase(int[] E, (int U, int V)[] aretes, double[] w, Comparaisons cmp, UnionFind uf)
{
    var arbre = new List<int>();
    if (E.Length == 0) return arbre;
    int maxPoids = E.Max(e => (int)w[e]);
    var seaux = new List<int>[maxPoids + 1];
    for (int k = 0; k <= maxPoids; k += 1) seaux[k] = new List<int>();
    foreach (int e in E) seaux[(int)w[e]].Add(e);
    for (int poids = 0; poids < seaux.Length; poids += 1)
    {
        foreach (int e in seaux[poids])
        {
            if (uf.Unir(aretes[e].U, aretes[e].V)) arbre.Add(e);
        }
    }
    return arbre;
}

// Jouet inspire de Karger-Klein-Tarjan (1995) : phases de Boruvka pour les sommets,
// echantillon + filtre pour les aretes, recursion sur ce qui reste.
List<int> MstEchantillonne(int[] E, (int U, int V)[] aretes, double[] w, Comparaisons cmp,
    UnionFind uf, Random rng, double p, int kPhases, double facteur,
    Func<int[], (int U, int V)[], double[], Comparaisons, UnionFind, List<int>> socle,
    List<(string Etape, int Prof, int NCur, int MCur, long Cmp)> trace, int depth, long[] parcours)
{
    var racines = new HashSet<int>();
    foreach (int e in E) { racines.Add(uf.Trouver(aretes[e].U)); racines.Add(uf.Trouver(aretes[e].V)); }
    int nCur = racines.Count;
    if (nCur <= 1) return new List<int>();
    if (E.Length <= facteur * nCur)
    {
        trace?.Add(("socle", depth, nCur, E.Length, cmp.N));
        return socle(E, aretes, w, cmp, uf);
    }
    trace?.Add(("entree", depth, nCur, E.Length, cmp.N));
    var arbre = new List<int>();
    for (int phase = 0; phase < kPhases; phase += 1)
    {
        arbre.AddRange(PhaseBoruvka(E, aretes, w, cmp, uf));
        E = E.Where(e => uf.Trouver(aretes[e].U) != uf.Trouver(aretes[e].V)).ToArray();
    }
    racines.Clear();
    foreach (int e in E) { racines.Add(uf.Trouver(aretes[e].U)); racines.Add(uf.Trouver(aretes[e].V)); }
    nCur = racines.Count;
    if (nCur <= 1) return arbre;
    trace?.Add(("boruvka", depth, nCur, E.Length, cmp.N));
    if (E.Length <= facteur * nCur)
    {
        trace?.Add(("socle", depth, nCur, E.Length, cmp.N));
        var fin = new List<int>(arbre);
        fin.AddRange(socle(E, aretes, w, cmp, uf));
        return fin;
    }
    var echantillon = E.Where(e => rng.NextDouble() < p).ToArray();
    var F = MstEchantillonne(echantillon, aretes, w, cmp, uf.Copie(), rng, p, kPhases, facteur, socle,
        trace, depth + 1, parcours);
    var chemin = new CheminMax(F, aretes, w, uf);
    var classes = Classifier(E, aretes, w, F, chemin, cmp, uf);
    if (parcours != null)
    {
        parcours[0] += chemin.Parcours;
        parcours[1] += chemin.Pas;
        parcours[2] += chemin.Verdicts;
    }
    var E2 = E.Where(e => classes[e] != "lourde").ToArray();
    trace?.Add(("filtre", depth, nCur, E2.Length, cmp.N));
    if (E2.Length >= E.Length)
    {
        trace?.Add(("socle", depth, nCur, E.Length, cmp.N));
        var fin = new List<int>(arbre);
        fin.AddRange(socle(E, aretes, w, cmp, uf));
        return fin;
    }
    var suite = new List<int>(arbre);
    suite.AddRange(MstEchantillonne(E2, aretes, w, cmp, uf, rng, p, kPhases, facteur, socle,
        trace, depth + 1, parcours));
    return suite;
}

int n38 = 4000;
var (aretes38, w38) = GrapheAleatoire(n38, 3.0, n38);
var trace38 = new List<(string Etape, int Prof, int NCur, int MCur, long Cmp)>();
var cmpT = new Comparaisons();
var parcoursT = new long[3];
var arbreT = MstEchantillonne(Enumerable.Range(0, aretes38.Length).ToArray(), aretes38, w38, cmpT,
    new UnionFind(n38), new Random(100), 0.5, 1, 1.5,
    (E, a, ww, c, u) => Kruskal(E, a, ww, c, u), trace38, 0, parcoursT);
var refT = Kruskal(Enumerable.Range(0, aretes38.Length).ToArray(), aretes38, w38,
    new Comparaisons(), new UnionFind(n38)).Select(e => w38[e]).OrderBy(x => x).ToArray();
Exiger(arbreT.Select(e => w38[e]).OrderBy(x => x).SequenceEqual(refT),
    "les poids de l'arbre rendu different de ceux de Kruskal");
Exiger(parcoursT[0] == parcoursT[1] - parcoursT[2], "Parcours != Pas - Verdicts");
var socles38 = trace38.Where(t => t.Etape == "socle").ToList();
Console.WriteLine($"Trace d'une execution (n = {n38}, m = {aretes38.Length}, densite 3) :");
Console.WriteLine($"   {trace38.Count} etapes, profondeur de recursion maximale {trace38.Max(t => t.Prof)}.");
Console.WriteLine("   L'indentation est la profondeur de recursion ; les comparaisons sont cumulees depuis le debut.");
foreach (var t in trace38.Take(26))
{
    string retrait = string.Concat(Enumerable.Repeat("| ", t.Prof));
    Console.WriteLine($"     {t.Etape,8} {retrait}composantes={t.NCur,6}  aretes={t.MCur,7}  comparaisons={t.Cmp,7}");
}
Console.WriteLine($"     ... ({trace38.Count - 26} autres lignes)");
Console.WriteLine("   etapes par type : " + string.Join(", ",
    trace38.GroupBy(t => t.Etape).Select(g => $"{g.Key}={g.Count()}")));
Console.WriteLine($"   socle : {socles38.Count} appels, sur au plus {socles38.Max(t => t.NCur)} composantes,"
    + $" {socles38.Sum(t => t.MCur)} aretes au total");
Console.WriteLine($"   instrument : {cmpT.N} comparaisons, soit {cmpT.N / (double)aretes38.Length:F2} par arete");
Console.WriteLine($"   parcours de chemin : {parcoursT[0]} comparaisons de poids entre aretes de F, non comptees"
    + $" par l'instrument, soit {parcoursT[0] / (double)aretes38.Length:F2} par arete");
Console.WriteLine($"   origine : {parcoursT[2]} verdicts, soit {parcoursT[2] / (double)aretes38.Length:F2} par arete ;"
    + $" longueur moyenne d'un chemin parcouru : {(double)parcoursT[1] / parcoursT[2]:F2} aretes de F");
Console.WriteLine($"   total : {cmpT.N + parcoursT[0]} comparaisons,"
    + $" soit {(cmpT.N + parcoursT[0]) / (double)aretes38.Length:F2} par arete");

var fonctionsSocle = new (string Nom, Func<int[], (int U, int V)[], double[], Comparaisons, UnionFind, List<int>> F)[]
{
    ("K", (E, a, ww, c, u) => Kruskal(E, a, ww, c, u)),
    ("R", (E, a, ww, c, u) => SocleTriParBase(E, a, ww, c, u)),
};
Console.WriteLine();
Console.WriteLine("   meme jouet, tailles croissantes (densite 3, trois graines, moyenne) :");
Console.WriteLine("   " + "n".PadLeft(6) + " " + "m".PadLeft(7) + " " + "Kruskal".PadLeft(8)
    + " " + "jouet/K".PadLeft(8) + " " + "parc/K".PadLeft(7) + " " + "K+parc".PadLeft(7)
    + " " + "jouet/R".PadLeft(8) + " " + "socle/m".PadLeft(8) + " " + "s".PadLeft(6));
var lignesCout = new List<(int N, int M, double V, double L)>();
foreach (int nG in new[] { 500, 1000, 2000, 4000, 8000, 16000 })
{
    var (aretesG, wG) = GrapheAleatoire(nG, 3.0, nG);
    int mG = aretesG.Length;
    var chrono = Stopwatch.StartNew();
    var cmpK = new Comparaisons();
    var refG = Kruskal(Enumerable.Range(0, mG).ToArray(), aretesG, wG, cmpK, new UnionFind(nG))
        .Select(e => wG[e]).OrderBy(x => x).ToArray();
    var valK = new List<long>();
    var valR = new List<long>();
    var valP = new List<long>();
    var valV = new List<long>();
    var valL = new List<double>();
    int socleAretes = 0;
    for (int graine = 0; graine < 3; graine += 1)
    {
        foreach (var f in fonctionsSocle)
        {
            var cmpX = new Comparaisons();
            var parcX = new long[3];
            var traceX = new List<(string Etape, int Prof, int NCur, int MCur, long Cmp)>();
            var arbreX = MstEchantillonne(Enumerable.Range(0, mG).ToArray(), aretesG, wG, cmpX,
                new UnionFind(nG), new Random(100 + graine), 0.5, 1, 1.5, f.F, traceX, 0, parcX);
            Exiger(arbreX.Select(e => wG[e]).OrderBy(x => x).SequenceEqual(refG),
                $"poids differents : {f.Nom} n={nG} graine={graine}");
            if (f.Nom == "K")
            {
                valK.Add(cmpX.N);
                valP.Add(parcX[0]);
                valV.Add(parcX[2]);
                valL.Add(parcX[1] / (double)parcX[2]);
                if (graine == 0) socleAretes = traceX.Where(t => t.Etape == "socle").Sum(t => t.MCur);
            }
            else
            {
                valR.Add(cmpX.N);
            }
        }
    }
    double moyK = valK.Average();
    double moyR = valR.Average();
    double moyP = valP.Average();
    double moyV = valV.Average();
    double moyL = valL.Average();
    lignesCout.Add((nG, mG, moyV / mG, moyL));
    Console.WriteLine($"   {nG,6} {mG,7} {cmpK.N / (double)mG,8:F2} {moyK / mG,8:F2} {moyP / mG,7:F2}"
        + $" {(moyK + moyP) / mG,7:F2} {moyR / mG,8:F2} {socleAretes / (double)mG,8:F2}"
        + $" {chrono.Elapsed.TotalSeconds,6:F1}");
}
Console.WriteLine();
Console.WriteLine("   d'ou vient la croissance (memes executions, trois graines, moyenne) :");
Console.WriteLine("   " + "n".PadLeft(6) + " " + "m".PadLeft(7) + " " + "verdicts/arete".PadLeft(14)
    + " " + "longueur moyenne".PadLeft(16) + " " + "comparaisons/verdict".PadLeft(20));
foreach (var l in lignesCout)
{
    Console.WriteLine($"   {l.N,6} {l.M,7} {l.V,14:F3} {l.L,16:F2} {l.L - 1,20:F2}");
}
Console.WriteLine();
Console.WriteLine("   densite variable (n = 4000, deux graines) :");
Console.WriteLine("   " + "m/n".PadLeft(4) + " " + "m".PadLeft(7) + " " + "Kruskal".PadLeft(8)
    + " " + "jouet/K".PadLeft(8) + " " + "parc/K".PadLeft(7) + " " + "K+parc".PadLeft(7)
    + " " + "jouet/R".PadLeft(8) + " " + "verdicts/arete".PadLeft(14) + " " + "longueur".PadLeft(8));
foreach (double ratioD in new[] { 3.0, 10.0, 30.0 })
{
    var (aretesR, wR) = GrapheAleatoire(4000, ratioD, 4000);
    int mR = aretesR.Length;
    var cmpKR = new Comparaisons();
    Kruskal(Enumerable.Range(0, mR).ToArray(), aretesR, wR, cmpKR, new UnionFind(4000));
    var valK = new List<long>();
    var valR = new List<long>();
    var valP = new List<long>();
    var valV = new List<long>();
    var valL = new List<double>();
    for (int graine = 0; graine < 2; graine += 1)
    {
        foreach (var f in fonctionsSocle)
        {
            var cmpX = new Comparaisons();
            var parcX = new long[3];
            MstEchantillonne(Enumerable.Range(0, mR).ToArray(), aretesR, wR, cmpX,
                new UnionFind(4000), new Random(200 + graine), 0.5, 1, 1.5, f.F, null, 0, parcX);
            if (f.Nom == "K")
            {
                valK.Add(cmpX.N);
                valP.Add(parcX[0]);
                valV.Add(parcX[2]);
                valL.Add(parcX[1] / (double)parcX[2]);
            }
            else
            {
                valR.Add(cmpX.N);
            }
        }
    }
    double moyK = valK.Average();
    double moyR = valR.Average();
    double moyP = valP.Average();
    double moyV = valV.Average();
    double moyL = valL.Average();
    Console.WriteLine($"   {ratioD,4:F0} {mR,7} {cmpKR.N / (double)mR,8:F2} {moyK / mR,8:F2} {moyP / mR,7:F2}"
        + $" {(moyK + moyP) / mR,7:F2} {moyR / mR,8:F2} {moyV / mR,14:F3} {moyL,8:F2}");
}

Trace d'une execution (n = 4000, m = 12000, densite 3) :


   77 etapes, profondeur de recursion maximale 5.


   L'indentation est la profondeur de recursion ; les comparaisons sont cumulees depuis le debut.


       entree composantes=  4000  aretes=  12000  comparaisons=      0


      boruvka composantes=  1005  aretes=   9001  comparaisons=  20000


       entree | composantes=  1004  aretes=   4479  comparaisons=  20000


... (47 outputs left out)

    m/n       m  Kruskal  jouet/K  parc/K  K+parc  jouet/R verdicts/arete longueur


      3   12000    12.31     4.77   16.67   21.43     4.76          1.340    13.44


     10   40000    14.04     5.40   21.63   27.03     5.40          1.690    13.80


     30  120000    15.62     5.61   25.55   31.16     5.61          1.802    15.18


### Lecture du résultat

**La trace, ligne à ligne.** L'exécution compte `77` étapes et descend au plus à la profondeur `5`. Les premières lignes montrent les deux mécanismes à l'œuvre :

- **Les phases de Borůvka réduisent les composantes vite.** `4000 → 1005 → 214 → 40 → 8`. La garantie classique est « every phase at least halves the number of vertices » ; sur ce graphe aléatoire, chaque phase divise par quatre environ. La première phase coûte `20000` comparaisons pour `12000` arêtes : moins de deux par arête.
- **L'échantillon réduit les arêtes.** À la profondeur 1, on entre avec `1004` composantes et `4479` arêtes — à peu près la moitié des `9001` d'après la phase de Borůvka, comme attendu avec `p = 1/2`.
- **Le filtre termine le travail en bas de la récursion.** Au plus profond, on entre avec `2` composantes et `4` arêtes ; le filtre n'en garde que `2`, et le socle conclut dès que le nombre d'arêtes tombe sous `1,5` fois le nombre de composantes.

Sur l'ensemble : `31` entrées, `19` phases de Borůvka, `19` filtres et `8` appels au socle.

**Le socle est négligeable.** Les `8` appels au socle portent sur au plus `8` composantes et `50` arêtes en tout — moins de `0,5 %` des `12000` arêtes. Les colonnes `jouet/K` et `jouet/R` diffèrent de `0,00` à `0,05` : donner au socle un tri par poids entier, hors modèle de comparaison, ne change pour ainsi dire rien. Le coût de ce jouet n'est donc **pas** dans le cas de base.

**Ce que voit l'instrument — et ce qu'il ne voit pas.** L'instrument `cmp` compte `56202` comparaisons, soit `4,68` par arête, contre `12,31` pour Kruskal sur le même graphe. Lu seul, ce chiffre ferait croire que le squelette coûte peu. Mais il ne voit pas les comparaisons internes des remontées de chemin (section 6). Elles sont à elles seules de `216599`, soit `18,05` par arête. Le coût complet, dans le modèle de comparaison de la section 1, est de `272801` comparaisons, soit `22,73` par arête — plus que Kruskal.

**D'où vient ce coût.** La ligne « origine » répond : `15727` verdicts, soit `1,31` par arête, pour une longueur moyenne de chemin de `14,77` arêtes de `F`. Un verdict coûte une comparaison vue par l'instrument, plus une par pas de remontée après le premier : ici `13,77` comparaisons internes par verdict en moyenne.

### Tailles croissantes : ce qui grandit, et ce qui ne grandit pas

| `n` | Kruskal | `jouet/K` | `parc/K` | `K+parc` | `K+parc` / Kruskal |
|---|---|---|---|---|---|
| `500` | `9,32` | `4,45` | `6,93` | `11,38` | `1,22` |
| `1000` | `10,31` | `4,53` | `9,08` | `13,61` | `1,32` |
| `2000` | `11,30` | `4,63` | `11,59` | `16,23` | `1,44` |
| `4000` | `12,31` | `4,73` | `18,49` | `23,22` | `1,89` |
| `8000` | `13,31` | `4,79` | `21,27` | `26,06` | `1,96` |
| `16000` | `14,30` | `4,84` | `32,43` | `37,27` | `2,61` |

- **Ce que voit l'instrument croît très peu** : `4,45 → 4,84` par arête en cinq doublements de `n`.
- **Les remontées de chemin croissent beaucoup** : `6,93 → 32,43`, soit `4,68` fois plus. D'un doublement à l'autre, le rapport va de `1,15` à `1,60` : la croissance est nette, mais irrégulière.
- **Le coût complet croît comme elles** : `11,38 → 37,27`, soit `3,27` fois. Kruskal, sur la même plage, passe de `9,32` à `14,30`, soit `1,53` fois. Le jouet coûte déjà plus que Kruskal à `n = 500` ; à `n = 16000`, il en coûte `2,61` fois plus.

**La seconde table dit où est la croissance.** Le nombre de verdicts par arête est presque stable : `1,181 → 1,382`. La longueur moyenne des chemins parcourus, elle, passe de `6,86` à `24,46` arêtes de `F`, soit `3,57` fois. Les deux facteurs se composent exactement : une comparaison par arête de `F` parcourue, moins la première qui n'en coûte pas, donne

`parc/K` = (verdicts par arête) × (longueur moyenne − 1).

Vérifié sur la dernière ligne : `1,382 × 23,46 = 32,42`, pour `32,43` mesuré — l'écart vient de l'arrondi des deux facteurs. Le filtre ne devient donc pas beaucoup plus fréquent quand `n` grandit ; il devient plus cher, parce que les chemins de `F` entre les extrémités d'une arête s'allongent.

**La densité joue dans le même sens.** À `n = 4000`, le coût complet passe de `21,43` à `27,03` puis `31,16` comparaisons par arête quand `m/n` passe de `3` à `10` puis `30` ; Kruskal passe de `12,31` à `14,04` puis `15,62`. Les verdicts par arête (`1,340 → 1,802`) et la longueur moyenne (`13,44 → 15,18`) augmentent tous les deux. La ligne de densité `3`, calculée sur deux autres graines, donne `16,67` pour `parc/K` là où la première table donne `18,49` : la variation d'un jeu de graines à l'autre reste faible devant la croissance mesurée.

**Ce que ce résultat enseigne sur le problème.** Réduire les sommets et appeler un socle ne coûtent presque rien. Ce qui coûte, c'est d'écarter les arêtes lourdes. Or le preprint distingue deux façons de s'y prendre. Traiter les requêtes de maximum de chemin **une à une**, au fil de l'eau — ce que fait la remontée de ce carnet —, a une borne inférieure : « Pettie [Pet06] ruled out one natural approach by proving that online verification, in which path-maximum queries arrive one at a time, indeed requires Ω(mα(m, n) + n) comparisons. » Une vérification ainsi traitée ne peut donc pas donner de garantie linéaire. Traiter toutes les requêtes **à la fois**, au contraire, se fait en temps linéaire : « an adaptation of the verification algorithm identifies all of them in linear time [DRT92, KKT95] ». C'est un vrai ingrédient d'algorithme, pas un détail d'implémentation — et il n'est pas implémenté ici, d'où la table.

### Ce que la table ne prouve pas

- **Elle ne prouve pas une loi de croissance.** Cinq doublements, une seule famille d'instances (graphes aléatoires de densité 3, trois graines), des rapports d'un doublement à l'autre irréguliers. Ce qui est mesuré : sur cette plage, le coût par arête du filtre naïf croît, et cette croissance vient de la longueur des chemins, pas du nombre de verdicts. Ce qui ne l'est pas : la forme de cette croissance.
- **Elle ne dit rien de l'algorithme du preprint.** Ce jouet tire son échantillon au hasard. L'algorithme de Khanna, Putterman et Song est **déterministe** : son échantillon et son filtre sont construits sans hasard, au prix d'une procédure de certificat (lemme 1.2) qui n'est pas implémentée ici.
- **Elle ne sépare pas le filtre des choix du jouet.** Probabilité `1/2`, une phase de Borůvka par niveau, seuil de socle `1,5` : d'autres choix déplaceraient les nombres.

Le résultat de cette section est donc net, et moins flatteur que l'instrument seul ne le laissait croire : le squelette Borůvka–échantillon–filtre **fonctionne** — l'arbre rendu a les mêmes poids que celui de Kruskal à chaque exécution, vérifié par `Exiger` — et son cas de base est négligeable ; mais avec un filtre naïf, il coûte plus de comparaisons que Kruskal, et l'écart grandit. Le problème difficile n'est pas le squelette : c'est d'y substituer un échantillonnage et un filtre **déterministes**. C'est ce que la section suivante décrit, par ses énoncés.

## 9. Ce que le preprint apporte

Les sections 2 à 8 ont mesuré des objets classiques : les comparaisons exactes de trois algorithmes, la complexité de décision de neuf petites topologies, la fonction `alpha` sous deux conventions, le coût de l'union-find, le lemme d'échantillonnage, et le prix d'un filtre naïf, qui croît sur la plage mesurée. Cette section dit ce que le preprint ajoute. Elle ne mesure rien : tout ce qui y est affirmé sur le preprint est **CITÉ**, recopié de la copie archivée du texte (rapport ECCC TR26-339, daté du 23 novembre 2026, lu le 26 novembre 2026 pour ce carnet). Aucune ligne de sa preuve n'est reproduite, vérifiée ni ré-exécutée ici.

### La question, et la réponse

Le §1 pose la question ainsi :

> « Does the MST problem admit a deterministic algorithm with O(m) worst-case running time? »

Et le premier théorème y répond :

> « Theorem 1.1. There is a uniform deterministic algorithm that, on every connected undirected multigraph with n vertices and m edges with real weights, finds an MST in O(m) worst-case time, accessing edge weights only through pairwise comparisons. »

Chaque mot de l'énoncé a un sens mesurable dans ce carnet :

- **uniform** : un seul algorithme pour toutes les tailles, pas une famille indexée ;
- **deterministic** : la borne vaut sur **toute** entrée, pas en espérance. La section 7 a chiffré l'écart sur le choix d'un échantillon : face à des poids construits contre lui, le choix aveugle laisse survivre `599` arêtes, le tirage aléatoire `198,22` en moyenne sur les mêmes poids ;
- **worst-case** : la même idée, dite autrement — pas de « avec forte probabilité » ;
- **only through pairwise comparisons** : le modèle de comparaison de la section 1, celui dans lequel l'instrument `Comparaisons` compte ;
- **connected** : le théorème 1.1 ne vise que les graphes connexes ; le corollaire 4.2, cité plus bas, étend aux forêts couvrantes minimales, donc aux graphes quelconques, en `O(m + n)`.

### Ce qui existait déjà, et pourquoi cela ne suffisait pas

Le §1 énumère trois résultats linéaires antérieurs, et aucun n'est un algorithme déterministe de construction dans le modèle de comparaison :

> « Naturally, the above question has seen significant attention, and as a consequence, several weaker linear-time bounds were already known. Klein and Tarjan [KT94], and later Karger, Klein, and Tarjan [KKT95], gave a randomized algorithm that always returns an MST and runs in linear time both in expectation and with high probability. Fredman and Willard [FW94] gave a deterministic linear-time algorithm for MST with word-sized integer weights on a word RAM. Similarly, deterministic linear-time verification of a supplied spanning tree was known by the early 1990s [DRT92]. »

Trois portes de sortie, chacune avec sa limite : le hasard (section 7), une machine à mots avec des poids entiers tenant dans un mot, ou la **vérification** — le classement exact dont la section 6 a mesuré le coût. Puis la borne qui tenait le record :

> « In this setting, the best previously known explicit worst-case bound for deterministic comparison-based construction was O(mα(m, n)), due to Chazelle [Cha00a]. »

La section 5 a calculé ce que vaut ce facteur : sous `m >= n`, `alpha <= 4` pour tout `n` inférieur à `2^65533`. Sur toute instance qu'une machine peut adresser, `m·alpha` et `m` diffèrent donc d'un facteur au plus 4 — visible comme constante, invisible comme asymptote. Le preprint ne promet pas un algorithme plus rapide sur les entrées réelles ; il ferme une question de borne. Le résumé le dit à sa manière : « providing the first improvement since the work of Chazelle (JACM 2000) ».

### La monnaie : un nombre de comparaisons

La section 4 a cité la définition de `T*(m, n)` et la réduction de Pettie et Ramachandran ; voici les bornes qui l'encadraient, et celle qui les remplace :

> « Consequently, a deterministic linear-time comparison-based MST algorithm exists if and only if T ∗(m, n) = O(m). However, this work of [PR02] left open a tight bound on T ∗(m, n). »

> « For m, n > 2, the best previously known bounds were T ∗(m, n) = Ω(m) and T ∗(m, n) = O(mα(m, n)), leaving open whether a linear number of comparisons always suffices. In this work, we show that it does: T ∗(m, n) = O(m). »

> « Theorem 4.1. Let T ∗(m, n) be the deterministic decision-tree complexity of the MSF problem on graphs with m edges and n vertices. Then T ∗(m, n) = O(m). »

> « Corollary 4.2. There exists a uniform deterministic O(m) time algorithm for computing the MST of any connected graph with m edges and n vertices, in both the RAM model and pointer machine model. In general, there is a deterministic O(m + n) time algorithm for computing the MSF of any graph (the extra O(n) time is necessary for even reading over all of the vertices). »

Le théorème 4.1 et le corollaire 4.2 ne disent pas la même chose, et l'ordre compte : **d'abord une borne sur les comparaisons**, **ensuite un algorithme**. Le pont entre les deux est antérieur au preprint, et le résumé l'annonce sans détour : « Our work proceeds by showing that the deterministic comparison complexity of the minimum spanning tree problem is linear, whereby a known reduction of Pettie and Ramachandran (JACM 2002) immediately implies a linear time algorithm. »

Le §2 donne ce pont sous forme de théorème, en renvoyant au théorème 5.8 de [PR02] :

> « Comparison complexity. Our analysis focuses on the number of edge weight comparisons rather than the running time. By the work of Pettie and Ramachandran [PR02], a deterministic comparison bound for MSF can be translated into a corresponding deterministic running time bound. »

> « Theorem 2.6 (Theorem 5.8 of [PR02]). Let T ∗(m, n) be the deterministic decision-tree complexity of the MSF problem on graphs with m edges and n vertices. There is a deterministic algorithm that computes the MSF of any graph with m edges and n vertices in O(T ∗(m, n)) time. »

> « Remark 2.7. As noted in [PR02], their algorithm does not require the full power of the RAM model and can be implemented in the pointer machine model of [Tar79b]. »

Composé avec le théorème 4.1, le théorème 2.6 remplace `O(T ∗(m, n))` par `O(m)` : c'est le corollaire 4.2.

### Décision contre algorithme : la même distinction, vue de l'histoire

La section 4 a séparé les deux objets : `T*(H)` est une profondeur d'arbre de décision — un nombre de comparaisons, sans coût pour **choisir** laquelle vient ensuite ; un algorithme doit aussi décider, en temps, quelle comparaison poser. L'histoire de la vérification, au §1, donne l'exemple le plus net de cet écart :

> « In his work [Tar79a], Tarjan showed how to solve this path-maximum problem in O(mα(m, n)) time using path compression on balanced trees. Komlós [Kom85] then refined this, and showed that a linear number of comparisons suffices for verification. However, his method did not give a linear-time way to decide which comparisons to make. This was then first achieved by Dixon, Rauch, and Tarjan [DRT92], who obtained a linear-time implementation on a unit-cost RAM by combining Komlós's algorithm with table lookup for small subproblems and with Tarjan's earlier method. King [Kin97] later gave a simpler implementation of this algorithm, and ultimately, Buchsbaum et al. [BKRW98, BGK+08] even achieved linear time on a pointer machine. »

Trois questions distinctes s'y lisent, dans l'ordre du texte :

1. **combien** de comparaisons suffisent : Komlós montre qu'un nombre linéaire suffit ;
2. **comment** les choisir en temps linéaire : Dixon, Rauch et Tarjan obtiennent une implémentation linéaire sur une RAM à coût unitaire, en combinant l'algorithme de Komlós à une consultation de tables pour les petits sous-problèmes et à la méthode antérieure de Tarjan ;
3. **sur quelle machine** : King donne une implémentation plus simple, puis Buchsbaum et al. atteignent le temps linéaire sur un modèle à pointeurs.

C'est exactement la lecture de `T*` faite à la section 4 : un arbre de décision optimal n'est pas un programme. Et la structure du résultat principal est la même : le preprint borne le nombre de comparaisons (théorème 4.1), et l'algorithme qui en découle est celui de Pettie et Ramachandran (corollaire 4.2).

Le même écart se laisse mesurer dans ce carnet. À chaque arête qu'il classe, le filtre de la section 8 demande le poids maximum d'un chemin dans la forêt `F`, et il y répond en parcourant ce chemin. Sur l'exécution tracée, `15727` questions ont coûté `216599` comparaisons, soit `13,77` par question, et ce prix croît avec la taille (section 8). Or le nombre de comparaisons nécessaires pour ces questions est connu — le §2 le rappelle :

> « Lemma 2.9 ([Kom85]). Given a forest F on n vertices and q query paths specified by their endpoints, the maximum edge weight on each query path can be determined deterministically using O(n + q) comparisons. »

Un nombre linéaire de comparaisons suffit donc, en la taille de la forêt et le nombre de questions ; le texte cité plus haut ajoute ce qu'il a fallu de plus pour en faire une méthode de temps linéaire. Le carnet s'arrête ici : la procédure de certificat du preprint, qui remplace ce filtre, n'est pas implémentée (section 11).

### L'ingrédient neuf : un certificat qui réduit le graphe

Le §1.2 nomme l'apport technique :

> « As alluded to above, the key new technical ingredient that we introduce is a mechanism for efficiently finding many “redundant edges” (i.e., edges which are the heaviest edges in a cycle in the graph). »

La notion d'arête **supportée**, citée à la section 6, est celle qui rend ce mécanisme énonçable. Il s'agit de produire une forêt `F` qui supporte **beaucoup** d'arêtes, sans avoir à les classer une par une. C'est l'objet du lemme 1.2 :

> « Lemma 1.2 (Certificate Procedure). Let G be an undirected multigraph with n vertices, m edges, and no isolated vertices. For every fixed constant ε > 0, there exists a deterministic certificate procedure that outputs a forest F ⊆ E(G). The procedure may invoke exact MSF calls on edge sets E1, . . . , Eℓ, each defined on a subset of V(G). Apart from these calls, the procedure performs O_ε(m) weight comparisons. Let U(F) ⊆ E(G) denote the set of edges unsupported by F. The procedure guarantees that » `|F| + |U(F)| + Σ_{i=1}^{ℓ} |Ei| <= ε·m + C_ε·√(nm)` « where Cε is a constant depending only on ε. »

Les mots sont ceux du texte ; l'inégalité est recomposée à partir des morceaux que l'extraction du PDF a dispersés sur plusieurs lignes (la racine porte sur `nm` ; les parenthèses sont ajoutées pour le dire en texte brut). Trois quantités entrent dans cette somme :

- `|F|`, la forêt produite, qu'il faut garder ;
- `|U(F)|`, les arêtes que `F` ne supporte pas : on ne peut pas les jeter, puisque l'une d'elles peut appartenir à l'arbre ;
- `Σ|Ei|`, la taille totale des sous-instances confiées à des appels exacts.

Le coût, lui, est de `O_ε(m)` comparaisons de poids **en dehors** de ces appels : linéaire, avec une constante qui dépend de `ε`. Les auteurs déplient le lemme ainsi :

> « Let us now unpack this lemma: we fix ε to be a small constant (in the final proof, we may even choose it to be 1/8). The lemma then guarantees that there is a procedure which produces a forest F which in fact supports many of the original edges in the graph. Because these original edges are supported, this then provides a proof that (unless they themselves are in the forest), they are not present in the MST. Viewing this as a filtering procedure, this means that we can reduce the number of edges in the graph from |E(G)| all the way down to |F| + |U(F)|. »

### Le terme en racine, et le rôle des phases de Borůvka

L'inégalité ne suffit pas encore. Si `n` est de l'ordre de `m`, le terme `C_ε·√(nm)` est de l'ordre de `m`, et plus rien ne décroît. Le §1.2 indique la parade ; la citation est coupée autour des deux formules, recomposées comme ci-dessus :

> « This expression itself is still not perfect however. On the right hand side, the ε · m term is quite small, but » `C_ε·√(nm)` « term can be quite large, especially when n approaches m. This is the key place where Borůvka phases come into play. A single Borůvka phase uses O(m) comparisons and halves the number of vertices in the graph. Because Cε is just a constant, we can then apply a large enough constant number of Borůvka phases to reduce n, and thereby ensure that » `C_ε·√(nm) <= m/4`.

C'est ici que la mesure de la section 3 prend son sens. Une phase de Borůvka coûte `O(m)` comparaisons, et la section en a compté de `5,38` à `7,69` par arête selon les tailles. Un **nombre constant** de ces phases divise `n` par une puissance de deux fixée, choisie assez grande pour que le terme en racine passe sous `m/4`. Les phases de Borůvka ne sont pas un ornement : elles achètent la décroissance du terme en racine, pour un coût linéaire.

### La récurrence, et pourquoi elle referme sur `O(m)`

La preuve du théorème 4.1 s'écrit en quatre gestes. Les phrases entre guillemets sont recopiées ; les formules sont recomposées comme ci-dessus.

1. **`k` phases de Borůvka.** « The algorithm begins by performing k Borůvka steps. This requires O(km) = O(m) comparisons. » Le nombre de sommets non isolés qui restent est alors au plus `n / 2^k`.
2. **Le certificat sur le graphe contracté.** « Otherwise, we apply the certificate procedure from Lemma 3.1 to G′. »
3. **Ce qui est transmis aux appels récursifs.** Le texte annonce « The total number of edges across all recursive MSF calls is bounded by », puis la chaîne
   `|F ∪ U(F)| + Σ_i |Ei|  <=  (1/8)·m + C_{1/8}·√(m·n′)  <=  (1/8)·m + (C_{1/8} / 2^(k/2))·√(m·n)  <=  (1/8)·m + (C_{1/8} / 2^(k/2))·√(2m²)  <=  (1/8)·m + (3/8)·m  =  m/2`.
   Trois passages la justifient : `n′ <= n / 2^k`, puis `n <= 2m` (aucun sommet isolé), puis le choix de `k`, que le texte fixe ainsi : « We set ε = 1/8, and choose a sufficiently large constant integer k such that » `C_{1/8} · 2^((1−k)/2) <= 3/8`.
4. **La récurrence.** Les auteurs posent `T(m) <= Σ_i T(m_i) + O(m)`, où `T(m)` est le nombre maximal de comparaisons pour une forêt couvrante minimale d'un graphe à `m` arêtes et `Σ_i m_i <= m/2`, puis concluent : « This yields T(m) = O(m), and completes the proof. »

Le dernier pas n'est pas détaillé par les auteurs. Il tient en une ligne, et c'est le seul calcul que cette section ajoute au texte. Si un appel de taille `m` ne transmet à ses sous-appels, **à eux tous**, qu'au plus `m/2`, alors la somme des tailles sur tout l'arbre de récursion est bornée par `m + m/2 + m/4 + ... = 2m`. Chaque appel coûte, hors ses sous-appels, un nombre de comparaisons proportionnel à sa propre taille ; le total est donc `O(m)`.

### L'histoire des bornes, en un tableau

Le tableau 1 du preprint situe le résultat. Ses onze premières lignes, ci-dessous, donnent pour des graphes simples connexes les bornes déterministes dans le pire cas, en modèle de comparaison ; la dernière est celle que le preprint ajoute.

| Année(s) | Algorithme | Comparaisons dans le pire cas |
|---|---|---|
| 1926 | Borůvka | `O(m log n)` |
| 1930 / 1957 | Jarník, puis Prim | `O(m + n log n)`, avec des tas de Fibonacci |
| 1956 | Kruskal | `O(m log n)` |
| 1975 | Yao | `O(m log log n)` |
| 1975 | Johnson | `O(m log^d n)` |
| 1976 | Cheriton et Tarjan | `O(m log(log^d n))` |
| 1984 / 1987 | Fredman et Tarjan | `O(m β(m, n))` |
| 1984 / 1986 | Gabow, Galil et Spencer, puis avec Tarjan | `O(m log β(m, n))` |
| 1997 | Chazelle | `O(m α(m, n) log α(m, n))` |
| 2000 | Chazelle | `O(m α(m, n))` |
| 2000 / 2002 | Pettie et Ramachandran | `O(T ∗(m, n))`, optimal |
| 2026 | Khanna, Putterman et Song (ce preprint) | `O(m)` |

La légende du tableau définit `d := 2 + m/n` et `beta(m, n)` comme le plus petit entier `i >= 1` pour lequel le `i`-ième logarithme itéré de `n` ne dépasse pas `m/n`. Le preprint ajoute deux résultats linéaires, hors du modèle de comparaison : Fredman et Willard (1990 / 1994), « O(m) for word-sized integer weights on a word RAM », et Klein puis Karger, Klein et Tarjan (1994 / 1995), « O(m) in expectation and with high probability ».

Les trois algorithmes de la section 3 figurent dans ce tableau : Borůvka (1926), Kruskal (1956) et Prim (1957, après Jarník en 1930). Les comptages en retrouvent l'allure. Kruskal paie un tri : `+1,00` comparaison par arête à chaque doublement de `n`. Borůvka paie ses phases : `4` à `6` sur les tailles mesurées. Prim, enfin, paie un tas : l'implémentation mesurée utilise un tas binaire écrit à la main, pas les tas de Fibonacci de la borne du tableau — c'est la borne du texte qui vaut `O(m + n log n)`, pas celle du code de la section 3.

### Où le preprint s'arrête, et où ce carnet s'arrête

- **La preuve n'est pas reproduite.** Ce carnet énonce le lemme 1.2 et recopie la trame de la preuve du théorème 4.1 ; il ne démontre ni l'un ni l'autre, et rien dans ses cellules ne les vérifie. La procédure de certificat elle-même, objet de la section 3 du preprint, n'est pas implémentée.
- **La constante n'est pas chiffrée.** Le texte lu donne `k` par l'inégalité de l'étape 3, sans valeur numérique pour `C_{1/8}` : la preuve du lemme se termine par « Taking γ ≥ 16/ε proves Lemma 3.1. », avec des paramètres fixés à `t = γ·d^3` et `r = 8·t^3`. En dérouler les définitions jusqu'à un nombre est un travail que ce carnet ne fait pas.
- **Le statut est celui d'un preprint.** Rapport ECCC TR26-339, daté du 23 novembre 2026, lu trois jours plus tard : ce n'est pas une publication évaluée par les pairs, et rien ici ne dit qu'il en deviendra une sans correction.

La section suivante rapporte un dernier fait, dans les mots des auteurs : l'origine qu'ils donnent eux-mêmes à ces idées.

## 10. Provenance IA : le texte des auteurs

Le preprint comporte une section 1.4 intitulée « AI Acknowledgments ». Elle est recopiée ici dans l'ordre, puis suivie du billet de blog du 24 novembre 2026 qui l'a reprise. Cette section n'ajoute aucun commentaire : ni éloge, ni réserve. Elle rapporte ce que le texte dit, et ce qu'il ne dit pas.

### Le §1.4 du preprint

> « A. P. would like to thank OpenAI for providing the credits to Astra 6.2 which were used in generating this result. »

> « Indeed, the main ideas in this paper were entirely generated by AI – specifically 6.2 Astra, on a mixture of Ultra and Max effort – with minimal human intellectual intervention (see below). »

> « The authors then worked to first carefully verify, and then to simplify and clarify the treatment, resulting in the present manuscript. »

Puis le récit de la genèse, dans leurs mots :

> « The authors had originally tried to query Astra 6 and 6.2 with a direct prompt of achieving deterministic linear comparison complexity for this problem, with no success. The authors then retried with the simpler task of proving any bound of the form o(mα), which would be an improvement over the work of Chazelle [Cha00a]. A team of 15 agents (managed across several Ultra and Max effort settings) succeeded in proving such an improvement after 15 hours, providing a bound »

La citation s'arrête là : la formule qui suit dans le texte est déchirée par l'extraction du PDF, et ce carnet ne la recompose pas. La suite reprend :

> « This gave the authors confidence that O(m) is indeed the right answer. Before that, there was some doubt that linear time was the correct answer, largely due to Ω(mα) lower bounds for related union-find data structures (see [Tar79b]), and variants of the MST problem [Pet06]. »

> « After this, the authors continuously prompted the model to iteratively improve the comparison complexity, resulting in a sequence of small improvements. This took several forms. The first improvement was a bound of the form O(m log(α)), which was subsequently followed by O(m log∗(α)), then O(mα(α)), and then O(mΓ(α)). »

Le texte présente alors `Γ` comme l'inverse d'une fonction hyper-ackermannienne, et précise :

> « In particular, Γ is smaller than any constant number of iterated inverse-Ackermann functions. This was then followed by bounds of the form O(m log(Γ(α))), then O(m log∗(Γ(α))), and even the absurdly small O(m · Γ(Γ(α))). »

> « The authors then sent off one final batch of runs which used the entire history of developed algorithms until this point, and asked for a single linear time algorithm, which was finally generated. »

Enfin la contribution humaine qu'ils désignent comme la plus grande :

> « Throughout the entire process leading up to the discovery of our main result, human intellectual intervention was limited. The largest such intervention was the advice that the authors did not believe it to be a fruitful direction to pursue “structural graph theory” approaches to the problem …, and repeatedly redirected attempts that got stuck in such pursuits. »

Le membre de phrase remplacé par des points de suspension définit ces approches : des algorithmes pour des graphes à structure particulière, de faible densité, sans petits cycles. Et le §1.3, juste avant, dit d'où la question est venue :

> « We thank Vijaya Ramachandran for drawing our attention to her joint work with Seth Pettie [PR02] during the first author's visit to the University of Texas at Austin in spring 2026. It was then that we began working on the problem of designing a deterministic linear-time MST algorithm. »

La réduction que le théorème 4.1 transforme en algorithme est donc aussi ce qui a mis les auteurs sur le problème ; le texte le dit sans détour.

### Ce que le texte ne dit pas

Quatre absences, aussi vérifiables que les présences :

- **Il ne dit pas que le résultat est évalué.** C'est un rapport ECCC, daté du 23 novembre 2026.
- **Il n'attribue pas la vérification au modèle.** La phrase est « The authors then worked to first carefully verify », et le texte ne quantifie ni le temps ni la répartition de ce travail.
- **Il ne donne pas de durée totale.** Le seul chiffre de durée est celui du premier résultat partiel — « after 15 hours » pour une borne en `o(mα)` — pas du résultat final.
- **Il ne dit pas qui a écrit quoi dans la preuve.** « minimal human intellectual intervention » n'est pas chiffré.

### Le billet du 24 novembre

Le billet de blog qui a signalé le résultat est celui de Scott Aaronson, intitulé « I've been drinking wine in my kitchen », daté du mardi 24 novembre 2026 à 17 h 17 et lu le 26 novembre 2026. Il situe d'abord le résultat :

> « yesterday Sanjeev Khanna, Aaron Putterman, and Junkai Song posted an ECCC preprint with a deterministic linear-time algorithm for finding minimum spanning trees in graphs. (The best previously-known algorithm was ridiculously close to linear time—n α(n) where α is the inverse of the Ackermann function—but not quite. A randomized linear-time algorithm was known.) This was a huge open problem for decades, and one that my wife Dana spent roughly two years working on. »

Puis il recopie la phrase de provenance, avec un mot entre crochets qui est de lui :

> « Khanna et al. write, “the main ideas in [our] paper were entirely generated by AI—specifically 6.2 Astra, on a mixture of Ultra and Max effort—with minimal human intellectual intervention. The authors then worked to first carefully verify, and then to simplify and clarify the treatment.” »

Le billet n'ajoute, de son propre cru, aucun commentaire sur la provenance : il rapporte le fait et poursuit. C'est la même sobriété qui est reprise ici. La provenance est un fait du texte ; elle est **CITÉ**, comme les énoncés de la section 9, et elle ne change rien à ce qui a été mesuré dans les sections 2 à 8.

## 11. Mesuré / Cité / Ouvert

Trois régimes de vérité, et une ligne par affirmation : ce que les cellules ont calculé, ce qui vient du texte du preprint ou du billet qui l'a repris, et ce que le carnet ne peut pas conclure.

### Mesuré

Toutes les valeurs de cette table sortent des cellules des sections 2 à 8, sur la machine qui a exécuté le carnet ; les colonnes de temps sont volontairement absentes.

| § | Objet mesuré | Valeur clé |
|---|---|---|
| 2 | Pire cas du `TriFusion` du carnet : toutes les permutations pour `n <= 8`, récurrence et forme close pour `9 <= n <= 400`, coût marginal pour `n <= 2000` | `C(n) = n·k - 2^k + 1` avec `k = ceil(log2 n)` ; `C(1000) = 8977` |
| 3 | Comparaisons de poids par arête, densité 3, `n` de 500 à 16000 | Kruskal `9,32 -> 14,30`, Prim `18,74 -> 29,84`, Borůvka `5,38 -> 7,69` ; phases `4 -> 6` ; gain de Kruskal par doublement `+0,99` à `+1,00` |
| 4 | `T*` exact par minimax, neuf topologies à six arêtes au plus | `T*` de `0` (étoile) à `6` (`K4`) ; `5` pour `C6` et `K23` ; tri fusion `11` sur les cinq topologies à six arêtes ; `76` comparaisons de tri contre `33` pour l'arbre optimal, soit `43` épargnées |
| 5 | `alpha` exacte, convention d'Ackermann-Peter, deux inégalités | `alpha <= 4` pour `m >= n` et `n <= 2^65533` ; `alpha >= 5` pour `n = 10^18`, `m = n/2` ; `2^65533` s'écrit en 19728 chiffres |
| 5 bis | Union-find de Kruskal (fractionnement), mêmes tailles | pas de compression par appel `1,322 -> 1,758` ; par arête `2,644 -> 3,515` |
| 6 | Règle fausse sur le triangle 0-1-2 | elle jette les 3 arêtes, dont les 2 de `F` : aucun arbre ne reste |
| 7 | Lemme d'échantillonnage, `n = 100`, 2000 tirages par réglage | rapport moyenne / borne `0,584 -> 0,991` ; tirages au-dessus de la borne : `214` à `m/n = 3` pour `p = 1/2` et `0` aux deux autres réglages, `817`, `844`, `713` à `m/n = 10` |
| 7 | Coût d'un filtre naïf, un tirage | `6836` comparaisons internes pour `901` arêtes comparées au maximum de leur chemin ; longueur moyenne `8,59` |
| 7 | Trois façons de choisir l'échantillon (`n = 100`, `m = 1000`) | aveugle : `0` pour choisir, `599` arêtes survivantes sur poids adverses contre `205` sur poids ordinaires ; aléatoire : `0`, `198,22` contre `198,17` ; tri : `4932` / `8732` pour choisir, `99` contre `99` |
| 8 | Jouet inspiré de KKT, `n = 4000`, `m = 12000`, exécution tracée | `77` étapes, profondeur de récursion `5`, `15727` verdicts, `272801` comparaisons soit `22,73` par arête, contre `12,31` pour Kruskal |
| 8 | Même jouet, tailles croissantes, trois graines | instrument seul `4,45 -> 4,84` par arête ; parcours de chemin `6,93 -> 32,43` ; total `11,38 -> 37,27` |
| 8 | D'où vient la croissance | verdicts par arête `1,181 -> 1,382` ; longueur moyenne d'un chemin `6,86 -> 24,46` |
| 8 | Densité variable, `n = 4000` | total `21,43` / `27,03` / `31,16` par arête pour `m/n` = 3 / 10 / 30, contre `12,31` / `14,04` / `15,62` pour Kruskal |

### Cité

Rien de ce qui suit n'a été calculé ici : chaque énoncé est recopié de la copie archivée du rapport ECCC TR26-339 (23 novembre 2026) ou du billet du 24 novembre 2026, et le contrôle des citations est passé sur le texte.

| Énoncé | Où il est cité |
|---|---|
| Théorème 1.1 : algorithme déterministe uniforme en `O(m)` pire cas, comparaisons par paires seulement | §9 |
| Théorème 2.6 (théorème 5.8 de [PR02]) et remarque 2.7 : la borne de comparaison devient un temps | §9 |
| Théorème 4.1 (`T*(m, n) = O(m)`) et corollaire 4.2 (algorithmes en `O(m)` et `O(m + n)`) | §9 |
| Lemme 1.2 puis 3.1 : la procédure de certificat, et son inégalité sur la taille de la forêt, celle des arêtes non supportées et la somme des tailles des appels récursifs | §9 |
| Dépliage du lemme 1.2 : les arêtes supportées sont écartées de l'arbre, le graphe se réduit à `\|F\| + \|U(F)\|` ; le terme en racine, et le nombre constant de phases de Borůvka qui le fait passer sous `m/4` | §9 |
| Trame de la preuve du théorème 4.1 : `k` phases, certificat sur le graphe contracté, somme des tailles transmises, récurrence `T(m) = O(m)` | §9 |
| Lemme 2.9 ([Kom85]) : maximum sur `q` chemins en `O(n + q)` comparaisons | §9 |
| Phase de Borůvka : chaque phase divise au moins par deux le nombre de sommets | §3, §8 |
| Arbre de comparaison et `T*` : minimum sur tous les arbres corrects, graphes non connexes compris | §4 |
| Pettie et Ramachandran : un algorithme uniforme à machine à pointeurs qui atteint la borne, et une réduction qui transforme une borne de comparaisons en temps linéaire | §4, §9 |
| `alpha`, fonction inverse d'Ackermann à deux paramètres, comme dans l'analyse de Tarjan pour Union-find [Tar75] ; borne `O(m·alpha(m, n))` due à Chazelle [Cha00a] | §5, §9 |
| Bornes inférieures `Ω(m·alpha)` pour des structures Union-find voisines [Tar79b] ; vérification en ligne écartée par Pettie [Pet06] | §5, §8, §10 |
| Arêtes F-lourdes et F-légères, et l'arête supportée par une forêt : les deux définitions jumelles du §1.2 et du §3 | §6 |
| Algorithmes randomisés en temps linéaire [KT94], puis [KKT95] : leurs phases de Borůvka et leur filtrage par échantillon | §7, §9 |
| Vérification : maximum de chemins par compression de chemins [Tar79a], linéarité des comparaisons [Kom85], implémentation simplifiée [Kin97], puis traitement par lot en temps linéaire [DRT92, KKT95] | §7, §8, §9 |
| État de l'art du §1 : [KT94], [KKT95], [FW94], [DRT92], [Cha00a], [BKRW98, BGK+08] | §10 |
| Tableau 1 : les bornes par année, de Borůvka à Pettie-Ramachandran | §9 |
| §1.3 et §1.4 : remerciement à Ramachandran, provenance IA du résultat | §10 |
| Billet de Scott Aaronson du 24 novembre 2026 : les deux paragraphes sur le preprint | §10 |

### Ouvert

1. **La preuve.** Aucune ligne du preprint n'est reproduite, vérifiée ni ré-exécutée ici. Le carnet cite des énoncés ; il ne les établit pas.
2. **La reproduction du résultat.** La procédure de certificat (lemme 1.2) n'est pas implémentée. Le jouet de la section 8 est inspiré de Karger-Klein-Tarjan : il filtre par un échantillon **aléatoire**, ce que le preprint remplace précisément par un mécanisme déterministe.
3. **La constante.** `C_ε` n'est jamais chiffrée dans le texte : les paramètres sont posés (`ε = 1/8`, `γ >= 16/ε`, `t = γ·d^3`, `r = 8·t^3`) et le nombre de phases de Borůvka est un entier constant choisi en fonction de `C_{1/8}`. Savoir si cette constante est petite reste hors d'atteinte de ce carnet.
4. **`T*(m, n)` au-delà des petits graphes.** Le minimax exact s'arrête à six arêtes, sur neuf topologies. Aucune valeur de `T*(m, n)` pour une taille supérieure n'est calculée ici.
5. **Une loi de croissance pour le filtre naïf.** Les rapports mesurés (parcours de chemin sur Kruskal : `×4,68` sur cinq doublements) ne donnent pas d'exposant, et le carnet n'en propose pas.
6. **Le statut du texte.** C'est un rapport ECCC daté du 23 novembre 2026, lu le 26 novembre 2026 pour ce carnet. Le carnet ne dit rien de son évaluation.

## 12. Exercices

Trois exercices, à compléter sans casser l'exécution : les stubs rendent `null` et affichent un message, le carnet reste exécutable de bout en bout.

### Exercice 1 — Borůvka, phase par phase

La section 3 a mesuré Borůvka avec un compteur global : un total de comparaisons, et un nombre de phases. Ici, on veut le détail **par phase**. Une phase choisit, pour chaque composante, l'arête de poids minimal qui en sort, l'ajoute à la forêt, puis contracte ; chaque phase réduit au moins de moitié le nombre de composantes non isolées.

Écrivez la fonction de sorte qu'elle rende l'arbre, le nombre de phases, et la **liste des comparaisons consommées par chaque phase**. La question à laquelle votre mesure doit répondre : les comparaisons par arête et par phase restent-elles bornées quand `n` grandit, ou le prix d'une phase dérive-t-il ?

Indice : il suffit de garder, pour chaque composante, un candidat courant — une comparaison par arête et par phase, quand le candidat est mis à jour au fil du balayage. Le résultat mesuré de la section 3 se lit dans ce sens, et votre compteur doit le retrouver.

In [16]:
// Renvoie (arbre, nombre de phases, comparaisons consommees par chaque phase).
// Question : les comparaisons par arete et par phase restent-elles bornees quand n grandit ?
(List<int> Arbre, int Phases, List<long> ComparaisonsParPhase) BoruvkaCompte(
    int[] E, (int U, int V)[] aretes, double[] w, Comparaisons cmp, UnionFind uf)
{
    // TODO etudiant
    return default;
}

Console.WriteLine("Exercice a completer");

Exercice a completer


### Exercice 2 — les pires cas du tri fusion, comptés

La section 2 a établi la forme close du pire cas du `TriFusion` du carnet, et l'a vérifiée par énumération pour `n <= 8`. Vous refaites ici l'énumération pour `n` de 4 à 8 : parcourez toutes les permutations, mesurez chacune avec `TriFusion` et un compteur `Comparaisons`, et rendez le pire nombre de comparaisons et **le nombre de permutations qui l'atteignent**.

Deux questions, auxquelles votre code répond : la permutation renversée est-elle un pire cas ? Et combien y a-t-il de pires cas pour chaque `n` — un seul, deux, davantage ?

Indice : la fonction `ToutesPermutations` de la section 1 suffit ; le poids d'une permutation peut être la permutation elle-même, puisque `TriFusion` ne compare que deux poids entre eux.

In [17]:
// Renvoie (pire nombre de comparaisons, nombre de permutations qui l'atteignent).
// Le tri fusion est mesure par TriFusion et Comparaisons. Question : la permutation
// renversee est-elle un pire cas ? Combien de pires cas y a-t-il pour n = 4..8 ?
(int Pire, int Nombre) PiresPermutations(int n)
{
    // TODO etudiant
    return default;
}

Console.WriteLine("Exercice a completer");

Exercice a completer


### Exercice 3 — choisir l'échantillon par la structure

La section 7 a comparé trois façons de choisir un échantillon d'arêtes : aveugle (gratuite, ruinable), aléatoire (gratuite, garantie en espérance), tri complet (garantie, mais payée d'un tri). Une quatrième voie reste ouverte : choisir par la **structure**, sans jamais lire un poids — les arêtes dont les extrémités ont le plus petit degré.

Écrivez la fonction qui rend les indices de `int(p * m)` arêtes, triées par `(deg(u) + deg(v), indice)`, en ne lisant que `aretes` et `n`. Puis mesurez-la contre la construction de poids adverses de la section 7 : combien d'arêtes survivent au filtre, sur les poids adverses et sur les poids ordinaires ?

La question : une information de structure suffit-elle à éviter le pire cas du choix aveugle ? Le tri des clés `(degré, indice)` ne compare pas de poids ; il ne coûte donc rien à l'instrument `Comparaisons`, et le coût du choix reste `0`.

Indice : les degrés se comptent en un balayage de `aretes`, puis chaque arête reçoit sa clé — un tri de clés entières, hors du modèle de comparaison de poids.

In [18]:
// Choisit int(p*m) aretes par degre croissant des extremites, SANS regarder les poids.
// Ordre deterministe (degre, puis indice). Question : une information de structure
// suffit-elle a eviter le pire cas du choix aveugle aux poids ?
int[] EchantillonParDegre((int U, int V)[] aretes, int n, double p)
{
    // TODO etudiant
    return null;
}

Console.WriteLine("Exercice a completer");

Exercice a completer


## Conclusion

| Étape | Ce que le carnet a construit | Mesure |
|---|---|---|
| §2 Le coût d'un tri | compteur exact, énumération pour `n <= 8`, récurrence jusqu'à 400, coût marginal jusqu'à 2000 | `C(n) = n·k - 2^k + 1`, `C(1000) = 8977` |
| §3 Trois algorithmes | Kruskal (union-find par fractionnement), Prim (tas binaire écrit à la main), Borůvka | par arête à `n = 500` : `9,32` / `18,74` / `5,38` ; à `n = 16000` : `14,30` / `29,84` / `7,69` |
| §4 Complexité de décision | minimax sur les arbres de comparaison, neuf topologies | `T*` de `0` à `6` ; sur les cinq topologies à six arêtes, `11` comparaisons pour le tri contre `33` au total pour l'arbre optimal |
| §5 La fonction `alpha` | Ackermann-Peter exact, deux conventions, seuils | `alpha <= 4` sous `m >= n` et `n <= 2^65533` ; `m < n` change la valeur (`alpha >= 5` pour `n = 10^18`, `m = n/2`) |
| §5 bis L'union-find | comptage des pas de compression | `1,322 -> 1,758` pas par appel, `2,644 -> 3,515` par arête |
| §6 Le piège | une règle locale fausse sur un triangle | elle jette les 3 arêtes, dont les 2 de la forêt |
| §7 L'échantillon | lemme mesuré, filtre naïf, trois choix | `817` tirages au-dessus de la borne sur 2000 à `m/n = 10`, `p = 1/2` ; `599` survivantes sur poids adverses contre `205` sur poids ordinaires, et `198,22` pour le hasard |
| §8 Un jouet KKT | phases de Borůvka, filtre par échantillon, compteurs séparés | `22,73` comparaisons par arête au total, contre `12,31` pour Kruskal |

Ce que le preprint ajoute ne se mesure pas ici : un énoncé (théorème 4.1), un pont vers l'algorithme (théorème 2.6, dû à Pettie et Ramachandran) et un ingrédient neuf (la procédure de certificat, lemme 1.2). Le carnet les cite, les situe contre ce qu'il a mesuré, et s'arrête là.

## À retenir

1. **La complexité de décision n'est pas un temps.** `T*(H)` compte des comparaisons dans le pire cas d'un arbre de décision ; un algorithme doit en plus décider, en temps, quelle comparaison poser. La vérification d'un arbre couvrant en donne l'exemple historique : un nombre linéaire de comparaisons suffit depuis Komlós, et l'implémentation linéaire — comment les choisir — a demandé un travail ultérieur (section 9).
2. **`alpha` est une constante sur toute instance adressable, et une fonction non bornée en théorie.** Sous `m >= n`, le carnet montre `alpha <= 4` tant que `n <= 2^65533`. La borne `O(m·alpha)` et la borne `O(m)` diffèrent donc d'un facteur au plus 4 en pratique, et diffèrent quand même : la seconde ferme une question que la première laissait ouverte (section 5).
3. **Un filtre gratuit n'existe pas, et c'est là que portait la question.** Choisir un échantillon sans lire les poids est gratuit ; en déterministe, un adversaire le ruine (`599` arêtes survivantes contre `205` sur des poids ordinaires) ; en aléatoire, il est garanti en espérance (`198,22`). Le preprint apporte la façon déterministe qui manquait (section 7).
4. **La borne en `alpha` est celle d'une structure, pas celle du problème.** L'union-find de Kruskal et le parcours de chemin du filtre naïf croissent avec la taille sur la plage mesurée ; le problème, lui, admet une linéarité en nombre de comparaisons (théorème 4.1, cité).
5. **Un compteur ne mesure que ce qu'on lui montre.** Dans le jouet de la section 8, l'instrument comptait `4,68` comparaisons par arête ; les parcours de chemin en faisaient `18,05` de plus, en comparant des poids d'arêtes de la forêt entre eux. Le total, `22,73`, dépasse Kruskal (`12,31`) — et c'est ce total, pas le premier chiffre, qui se compare au théorème 1.1.
6. **La provenance d'un résultat se cite comme un fait du texte, sans amplification.** Le §1.4 du preprint dit que les idées principales ont été générées par IA, avec une intervention humaine intellectuelle minimale, puis vérifiées, simplifiées et clarifiées par les auteurs (section 10). Ce que cette phrase ne dit pas — ce que vaut la vérification, qui a écrit quoi — n'est pas dans le texte, et le carnet ne le comble pas.

## Pour aller plus loin

- **Carnet jumeau** : [Complexity-03k — version Python](Complexity-03k-KhannaPuttermanSong-ArbreCouvrantLineaireDeterministe-Python.ipynb) — les mêmes expériences, compteurs compris, avec les figures tracées par matplotlib.
- **L'arbre couvrant comme base d'un matroïde** : [Complexity-04d — la secrétaire matroïdale](Complexity-04d-Secretaire-Matroidal-CSharp.ipynb) — le même objet, du côté des matroïdes.
- **La fonction d'Ackermann** : [Complexity-07b — le tour des grands nombres](Complexity-07b-LeTourDesGrandsNombres-CSharp.ipynb) — où `alpha` trouve sa place parmi les fonctions à croissance rapide.
- **Le socle de la série** : [Complexity-03 — la hiérarchie temporelle](Complexity-03-HierarchieTemporelle-CSharp.ipynb) — dont ce carnet est une accrétion.
- **Source principale** : Sanjeev Khanna, Aaron Putterman, Junkai Song, *A Deterministic Linear Time Algorithm for Minimum Spanning Tree*, ECCC Report TR26-339 (23 novembre 2026), [eccc.weizmann.ac.il/report/2026/339](https://eccc.weizmann.ac.il/report/2026/339/) — copie archivée au gisement `Bibliographie IA/Automata/Complexity Theory and Hardness` (PDF et notice ECCC). Toute attribution du carnet se lit contre cette copie.
- **Source secondaire** : Scott Aaronson, *I've been drinking wine in my kitchen*, billet du 24 novembre 2026, [scottaaronson.blog/?p=10362](https://scottaaronson.blog/?p=10362) — copie HTML archivée au même rayon.

**Navigation** : [Index](README.md) | [Version Python](Complexity-03k-KhannaPuttermanSong-ArbreCouvrantLineaireDeterministe-Python.ipynb) | [Carnet 03 de base](Complexity-03-HierarchieTemporelle-CSharp.ipynb)